# FreshMatch · Smarter swaps, fewer empty baskets
## AI-Powered Product Substitution Discovery and Intelligent Alternative Recommendation Using Machine Learning
**K. Darshan Sai** · RA2311027010002  
**J. Ranga Mokshagna** · RA2311027010003  
**P. Bharat Sai** · RA2311027010007

Run **Runtime → Run all** in Google Colab (CPU). This notebook installs dependencies, obtains data, mines complements, learns substitution suitability, evaluates against baselines, saves artifacts, writes the entire app, and launches a public preview. No separate Python source file is required. Typical compute target: ten minutes; package installation, Kaggle transfer, and tunnel availability depend on the network.

**Scientific boundary.** Co-purchases are usually complements, not substitutes. Weak labels encode a heuristic, so classifier accuracy measures agreement with that heuristic. A separate, chronologically held-out next-order benchmark measures whether recommendations contain actual subsequent purchases. Neither benchmark proves causal sales recovery. Synthetic data demonstrate the pipeline, not real-world effectiveness.

**Model comparison update:** RF and GB remain the proposed methods. LR, XGBoost, MLP and soft voting are comparison methods. A shared split/CV protocol selects deployment by training average precision; a joint held-out metrics/time table is reproduced in Model Performance.


In [1]:
%pip -q install "numpy==2.4.6" "pandas==2.3.3" "scipy==1.17.1" "scikit-learn==1.9.0" "mlxtend==0.23.4" "plotly==6.9.0" "streamlit==1.66.1" "matplotlib==3.11.1" "joblib==1.5.3" "requests==2.34.2" "kaggle==1.8.4" "nbformat==5.11.1" "nbclient==0.11.0" "ipykernel==6.31.0" "xgboost==3.2.0" "threadpoolctl==3.6.0"


Using installed pinned environment


### 1 · Reproducibility and run controls
The catalogue is bounded at 320 products and mining at length two to keep exact Apriori and FP-Growth tractable on free CPU. Both mine the same data at the same support. Set `FRESHMATCH_DATA_DIR` to a folder with extracted Instacart CSVs, or provide Kaggle credentials before running. The default attempts an authenticated download only when credentials exist; otherwise it automatically creates 288 products and 27,000 realistic orders. Instacart does not include prices or explicit brands: estimated prices and name-derived brands are labelled as such.

In [2]:
import os, sys, json, time, random, hashlib, warnings, subprocess, zipfile, shutil
from pathlib import Path
from time import perf_counter
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
from sklearn.inspection import permutation_importance
from threadpoolctl import threadpool_limits
from xgboost import XGBClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, confusion_matrix, roc_curve, precision_recall_curve, brier_score_loss)
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
import joblib
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown

SEED = 42
random.seed(SEED); np.random.seed(SEED)
START = perf_counter()
ROOT = Path.cwd()
ART = ROOT / 'artifacts'; ART.mkdir(exist_ok=True)
CACHE = ROOT / '.cache_freshmatch'; CACHE.mkdir(exist_ok=True)
DATA_DIR = Path(os.environ.get('FRESHMATCH_DATA_DIR', str(ROOT / 'instacart')))
MAX_PRODUCTS = 320
MIN_SUPPORT = .0015
COLOR = '#137C66'
px.defaults.template = 'plotly_white'
px.defaults.color_discrete_sequence = [COLOR, '#F0AE55', '#6B83B4', '#D47777']
pd.set_option('display.max_colwidth', 70)
print('Seed:', SEED, '| Working directory:', ROOT)

Seed: 42 | Working directory: C:\AML


### 2 · Data ingestion with an automatic, transparent fallback
Original data: [Instacart Market Basket Analysis on Kaggle](https://www.kaggle.com/c/instacart-market-basket-analysis/data). Competition access may require accepting its rules. Existing CSVs are preferred. Download errors and missing credentials trigger synthetic generation; no access restrictions are bypassed.

The generator includes household tastes, four shopping missions, unequal brand preferences, realistic prices, repeat purchases, and hidden need groups. A basket normally contains at most one item per need group. Complementary categories share shopping missions. Hidden need IDs are used **only for synthetic evaluation and data generation**, never as predictive features or labels. Each of 3,000 households has eight history orders and one future order.

In [3]:
def find_instacart(directory):
    required = ['orders.csv','order_products__prior.csv','products.csv','aisles.csv','departments.csv']
    if not directory.exists():
        return None
    for candidate in [directory] + sorted(p for p in directory.iterdir() if p.is_dir()):
        if all((candidate / f).exists() for f in required):
            return candidate
    return None

source_dir = find_instacart(DATA_DIR)
fallback_reason = ''
has_kaggle = bool(os.environ.get('KAGGLE_API_TOKEN') or
    (os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY')) or
    (Path.home()/'.kaggle'/'kaggle.json').exists())
if source_dir is None and has_kaggle:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    try:
        result = subprocess.run([sys.executable,'-m','kaggle.cli','competitions','download',
            '-c','instacart-market-basket-analysis','-p',str(DATA_DIR)],
            capture_output=True, timeout=90)
        if result.returncode != 0:
            raise RuntimeError('Kaggle access unavailable; check credentials and competition acceptance.')
        # Extract only expected CSV payloads, including the nested CSV zip layout.
        allowed = {'orders.csv','order_products__prior.csv','products.csv','aisles.csv','departments.csv'}
        for archive in list(DATA_DIR.glob('*.zip')):
            with zipfile.ZipFile(archive) as z:
                for name in z.namelist():
                    leaf = Path(name).name
                    if leaf in allowed or leaf in {x+'.zip' for x in allowed}:
                        with z.open(name) as src, open(DATA_DIR/leaf,'wb') as dst:
                            shutil.copyfileobj(src,dst)
        for archive in list(DATA_DIR.glob('*.csv.zip')):
            with zipfile.ZipFile(archive) as z:
                for name in z.namelist():
                    if Path(name).name in allowed:
                        with z.open(name) as src, open(DATA_DIR/Path(name).name,'wb') as dst:
                            shutil.copyfileobj(src,dst)
        source_dir = find_instacart(DATA_DIR)
    except (subprocess.TimeoutExpired, OSError, RuntimeError, zipfile.BadZipFile) as exc:
        fallback_reason = str(exc)
if source_dir is None and not fallback_reason:
    fallback_reason = 'No extracted Instacart CSVs or configured Kaggle credentials found.'

def synthetic_data():
    rng = np.random.default_rng(SEED)
    specs = [
      ('Milk','Dairy',['Whole milk','Low fat milk','Oat drink','Almond drink'],3.4,0),
      ('Cereal','Breakfast',['Rolled oats','Granola','Corn flakes','Bran flakes'],4.6,0),
      ('Bread','Bakery',['Whole wheat loaf','Rye loaf','Sourdough loaf','White loaf'],3.5,0),
      ('Coffee','Beverages',['Ground coffee','Espresso beans','Decaf ground coffee','Decaf instant coffee'],7.8,0),
      ('Pasta','Pantry',['Spaghetti','Linguine','Penne','Fusilli'],2.6,1),
      ('Sauce','Pantry',['Tomato basil sauce','Marinara sauce','Pesto sauce','Creamy basil sauce'],4.0,1),
      ('Protein','Chilled',['Chicken breast','Turkey slices','Firm tofu','Black bean patties'],6.5,1),
      ('Vegetables','Produce',['Baby spinach','Mixed greens','Broccoli florets','Green beans'],3.2,1),
      ('Snacks','Snacks',['Sea salt crisps','Corn tortilla chips','Rice crackers','Roasted nuts'],3.6,2),
      ('Dips','Chilled',['Mild salsa','Guacamole','Classic hummus','Yogurt dip'],4.2,2),
      ('Fruit','Produce',['Red apples','Green apples','Cavendish bananas','Navel oranges'],3.0,3),
      ('Yogurt','Dairy',['Plain Greek yogurt','Vanilla Greek yogurt','Strawberry yogurt','Coconut yogurt'],4.1,3)]
    brands = ['Harvest','Everyday','Green Valley','Orchard','Purely','Market House']
    rows=[]
    for c,(cat,dept,needs,base,mission) in enumerate(specs):
        for g,need in enumerate(needs):
            for b,brand in enumerate(brands):
                rows.append(dict(product_id=len(rows), name=f'{brand} {need}', category=cat,
                    aisle=cat, department=dept, brand=brand,
                    price=round(base*(.76+.115*b)*rng.uniform(.90,1.10),2),
                    need_id=c*2+g//2, mission=mission))
    products=pd.DataFrame(rows)
    lines=[]; orders=[]; last_seen={}
    brand_weights=np.array([.25,.24,.18,.14,.11,.08])
    for user in range(3000):
        # A shared household/occasion preference leaves a recoverable context signal.
        # Two synonymous product forms per need prevent exact-name matching from being an oracle.
        preference=float(rng.beta(.6,.6))
        form_preferences=rng.beta(2,2,size=12)
        favorite=int(rng.choice(6,p=brand_weights))
        mission_weights=rng.dirichlet([2,2,1.4,1.8])
        for number in range(1,10):
            oid=user*9+number
            orders.append((oid,user,number))
            mission=int(rng.choice(4,p=mission_weights))
            active=[c for c,s in enumerate(specs) if s[4]==mission or rng.random()<.18]
            style=int(rng.random()<preference)
            for c in active:
                need_group=style if rng.random()<.86 else 1-style
                form=int(rng.random()<form_preferences[c])
                g=need_group*2+form
                probs=brand_weights.copy(); probs[favorite]+= .45; probs/=probs.sum()
                b=int(rng.choice(6,p=probs))
                pid=c*24+g*6+b
                lines.append((oid,pid,int((user,pid) in last_seen)))
                last_seen[user,pid]=1
    return products,pd.DataFrame(orders,columns=['order_id','user_id','order_number']),pd.DataFrame(lines,columns=['order_id','product_id','reordered'])

def instacart_data(directory):
    raw=pd.read_csv(directory/'products.csv').merge(pd.read_csv(directory/'aisles.csv'),on='aisle_id').merge(pd.read_csv(directory/'departments.csv'),on='department_id')
    orders=pd.read_csv(directory/'orders.csv',usecols=['order_id','user_id','order_number','eval_set'])
    orders=orders[orders.eval_set=='prior'].copy()
    # Whole households, deterministically sampled. Every retained future order is truly later.
    users=np.sort(orders.groupby('user_id').size().loc[lambda x:x>=4].index)
    chosen=np.random.default_rng(SEED).choice(users,min(3000,len(users)),replace=False)
    orders=orders[orders.user_id.isin(chosen)].copy()
    last=orders.groupby('user_id').order_number.transform('max')
    historic_ids=set(orders.loc[orders.order_number<last,'order_id'])
    wanted=set(orders.order_id)
    chunks=[]
    for chunk in pd.read_csv(directory/'order_products__prior.csv',usecols=['order_id','product_id','reordered'],chunksize=1_000_000):
        chunks.append(chunk[chunk.order_id.isin(wanted)])
    lines=pd.concat(chunks,ignore_index=True)
    # Catalogue selection uses history only; do not choose products using future frequency.
    counts=lines[lines.order_id.isin(historic_ids)].product_id.value_counts()
    top=set(counts.head(MAX_PRODUCTS).index)
    products=raw[raw.product_id.isin(top)].rename(columns={'product_name':'name','aisle':'category'}).copy()
    products['aisle']=products['category']
    products['brand']=products['name'].str.split().str[0].fillna('Unknown')
    rng=np.random.default_rng(SEED)
    products=products.sort_values('product_id').reset_index(drop=True)
    products['price']=np.round(rng.lognormal(1.35,.4,len(products)),2)
    return products, orders[['order_id','user_id','order_number']], lines[lines.product_id.isin(top)]

source = 'Instacart' if source_dir else 'Synthetic grocery simulation'
fingerprint = 'v6-' + (str(source_dir.resolve()) + '-' + str((source_dir/'order_products__prior.csv').stat().st_mtime_ns) if source_dir else 'synthetic-42')
cache_file=CACHE/(hashlib.sha256(fingerprint.encode()).hexdigest()[:16]+'.joblib')
if cache_file.exists():
    products, orders, lines = joblib.load(cache_file)
else:
    products, orders, lines = instacart_data(source_dir) if source_dir else synthetic_data()
    joblib.dump((products,orders,lines),cache_file)
products=products.drop_duplicates('product_id').copy()
for col in ['name','category','aisle','department','brand']:
    products[col]=products[col].fillna('Unknown').astype(str).str.strip().replace('','Unknown')
products['price']=pd.to_numeric(products.price,errors='coerce').fillna(products.price.median()).clip(lower=.1)
lines=lines.dropna(subset=['order_id','product_id']).drop_duplicates(['order_id','product_id'])
lines=lines[lines.product_id.isin(products.product_id)]
lines['reordered']=pd.to_numeric(lines.reordered,errors='coerce').fillna(0).clip(0,1)
orders=orders.drop_duplicates('order_id').dropna(subset=['user_id','order_number'])
last_number=orders.groupby('user_id').order_number.transform('max')
history_orders=orders[orders.order_number<last_number].copy()
future_orders=orders[orders.order_number==last_number].copy()
history=lines[lines.order_id.isin(history_orders.order_id)].copy()
future=lines[lines.order_id.isin(future_orders.order_id)].copy()
products=products.sort_values('product_id').reset_index(drop=True)
products['idx']=np.arange(len(products))
pid_to_idx=dict(zip(products.product_id,products.idx))
print(f'DATA SOURCE: {source}')
if fallback_reason: print('Fallback reason:',fallback_reason)
print(f'{len(products):,} products | {len(history_orders):,} history baskets | {len(future_orders):,} held-out future baskets')
assert not set(history.order_id) & set(future.order_id)

DATA SOURCE: Synthetic grocery simulation
Fallback reason: No extracted Instacart CSVs or configured Kaggle credentials found.
288 products | 24,000 history baskets | 3,000 held-out future baskets


### 3 · Exploratory analysis and the basket matrix
Orders, not product rows, form the denominator of support. Empty retained baskets remain in that denominator. All price, popularity, reorder, vocabulary, and co-purchase features are fitted on history only. Reorder rate is the fraction of historical purchases marked as repeats; synthetic repeat flags are generated from each household's past.

In [4]:
order_to_row={v:i for i,v in enumerate(history_orders.order_id)}
rows=history.order_id.map(order_to_row).to_numpy()
cols=history.product_id.map(pid_to_idx).to_numpy()
B=sparse.csr_matrix((np.ones(len(history),dtype=np.float32),(rows,cols)),shape=(len(history_orders),len(products)))
B.data[:]=1
counts=np.asarray(B.sum(axis=0)).ravel()
products['popularity']=counts.astype(int)
products['support']=counts/len(history_orders)
rates=history.groupby('product_id').reordered.mean()
products['reorder_rate']=products.product_id.map(rates).fillna(0)
display(products[['name','category','price','popularity','reorder_rate']].head())
display(Markdown(f'**{source}** · {B.shape[0]:,} historical baskets · {B.shape[1]:,} products · matrix density {B.nnz/np.prod(B.shape):.2%}'))
fig=px.bar(products.groupby('category',as_index=False).popularity.sum(),x='category',y='popularity',title='Historical purchases by category',labels={'popularity':'Item purchases','category':'Category'})
fig.show()
px.histogram(products,x='price',color='department',title='Catalogue price distribution (USD)',labels={'price':'Price (USD)'}).show()
basket_sizes=np.asarray(B.sum(axis=1)).ravel()
px.histogram(x=basket_sizes,nbins=20,title='Historical basket sizes',labels={'x':'Items in retained catalogue','y':'Baskets'}).show()

,name,category,price,popularity,reorder_rate
0,Harvest Whole milk,Milk,2.73,621,0.157810
1,Everyday Whole milk,Milk,2.94,575,0.132174
2,Green Valley Whole milk,Milk,3.61,464,0.157328
3,Orchard Whole milk,Milk,3.91,353,0.110482
4,Purely Whole milk,Milk,3.81,267,0.134831


**Synthetic grocery simulation** · 24,000 historical baskets · 288 products · matrix density 1.63%

### 4 · Mine complements with Apriori AND FP-Growth
Both algorithms enumerate all frequent singletons and pairs at support 0.0015. `max_len=2` intentionally bounds the experiment. Rules include support, confidence and lift; lift above one means more co-purchase than independence predicts. Confidence is directional. Apriori and FP-Growth should find identical frequent sets and rules, although runtime differs. Basket rows and item columns are identical for the comparison.

In [5]:
basket_df=pd.DataFrame.sparse.from_spmatrix(B.astype(bool),columns=products.idx)
mine_key=hashlib.sha256((fingerprint+str(MIN_SUPPORT)+str(B.shape)).encode()).hexdigest()[:16]
mining_file=CACHE/f'mining-{mine_key}.joblib'
if mining_file.exists():
    mined, rules, mining_compare=joblib.load(mining_file)
    print('Using cached mining results; runtimes below are from the original computation.')
else:
    mined={}; rule_parts=[]; records=[]
    for name,fn in [('Apriori',apriori),('FP-Growth',fpgrowth)]:
        t=perf_counter()
        itemsets=fn(basket_df,min_support=MIN_SUPPORT,use_colnames=True,max_len=2)
        if any(len(s)>1 for s in itemsets.itemsets):
            rr=association_rules(itemsets,metric='confidence',min_threshold=.025)
            rr=rr[(rr.antecedents.map(len)==1)&(rr.consequents.map(len)==1)].copy()
        else:
            rr=pd.DataFrame(columns=['antecedents','consequents','support','confidence','lift'])
        duration=perf_counter()-t
        mined[name]=itemsets
        for _,r in rr.iterrows():
            rule_parts.append(dict(algorithm=name,a=int(next(iter(r.antecedents))),b=int(next(iter(r.consequents))),support=float(r.support),confidence=float(r.confidence),lift=float(r.lift)))
        records.append(dict(algorithm=name,seconds=duration,itemsets=len(itemsets),rules=len(rr)))
    rules=pd.DataFrame(rule_parts,columns=['algorithm','a','b','support','confidence','lift'])
    mining_compare=pd.DataFrame(records)
    joblib.dump((mined,rules,mining_compare),mining_file)
assert set(mined['Apriori'].itemsets)==set(mined['FP-Growth'].itemsets)
display(mining_compare)
px.bar(mining_compare,x='algorithm',y='seconds',text='rules',title='Association mining runtime · labels show rule counts',labels={'seconds':'Elapsed seconds','algorithm':'Algorithm'}).show()
px.bar(mining_compare,x='algorithm',y='rules',title='Rule counts at identical support and confidence',labels={'rules':'Directed rules','algorithm':'Algorithm'}).show()

Using cached mining results; runtimes below are from the original computation.


C:\Users\bhara\AppData\Local\Temp\ipykernel_43124\1659830409.py:1: FutureWarning: Allowing arbitrary scalar fill_value in SparseDtype is deprecated. In a future version, the fill_value must be a valid value for the SparseDtype.subtype.
  basket_df=pd.DataFrame.sparse.from_spmatrix(B.astype(bool),columns=products.idx)


,algorithm,seconds,itemsets,rules
0,Apriori,5.293857,676,776
1,FP-Growth,18.673200,676,776


### 5 · Features: shared need without shared basket
TF-IDF captures name similarity. A product's **co-purchase context** is its positive association profile with every other item, not its direct overlap with the candidate. Cosine similarity of these profiles can be high for alternatives that never co-occur. Direct lift uses add-one smoothing and a capped log transform. Ratios are symmetric (smaller / larger), keeping unordered pair groups intact.

Weak positives: same category, context similarity in the category's upper 35%, and lift < 0.55. Negatives: sampled cross-category pairs, strong complements, and within-category pairs with low context or high lift. Ambiguous pairs are excluded. This is heuristic supervision, not expert annotation; selection bias makes its test metrics optimistic. Need IDs and future orders are excluded from all features and labels.

In [6]:
C=(B.T @ B).toarray().astype(np.float64)
np.fill_diagonal(C,0)
expected=np.outer(counts,counts)/B.shape[0]
lift=(C+1)/(expected+1)
np.fill_diagonal(lift,0)
context_profile=np.log1p(np.maximum(lift-1,0))*np.sqrt(C)
context=cosine_similarity(context_profile).astype(np.float32)
tfidf=TfidfVectorizer(ngram_range=(1,2),min_df=1,strip_accents='unicode')
text_vectors=tfidf.fit_transform(products.name.str.lower())
text_sim=cosine_similarity(text_vectors).astype(np.float32)
cat=products.category.to_numpy(); aisle=products.aisle.to_numpy()
dept=products.department.to_numpy(); brand=products.brand.to_numpy()
prices=products.price.to_numpy(); pop=products.popularity.to_numpy(); reorder=products.reorder_rate.to_numpy()
FEATURES=['same_category','same_aisle','same_department','same_brand','text_similarity','context_similarity','log_lift','price_ratio','popularity_ratio','reorder_similarity']
def pair_features(a,b):
    a=np.asarray(a,dtype=int); b=np.asarray(b,dtype=int)
    return np.column_stack([cat[a]==cat[b],aisle[a]==aisle[b],dept[a]==dept[b],brand[a]==brand[b],
        text_sim[a,b],context[a,b],np.log1p(np.minimum(lift[a,b],20)),
        np.minimum(prices[a],prices[b])/np.maximum(prices[a],prices[b]),
        (np.minimum(pop[a],pop[b])+1)/(np.maximum(pop[a],pop[b])+1),1-np.abs(reorder[a]-reorder[b])]).astype(np.float32)

a,b=np.triu_indices(len(products),1)
pair_table=pd.DataFrame({'a':a,'b':b})
same=cat[a]==cat[b]
thresholds={c:np.quantile(context[a[(cat[a]==c)&same],b[(cat[a]==c)&same]],.65)
            for c in np.unique(cat) if np.sum((cat[a]==c)&same)>0}
threshold=np.array([thresholds.get(c,1.) for c in cat[a]])
positive=same & (context[a,b]>=threshold) & (lift[a,b]<.55)
complement=(lift[a,b]>1.5)&(C[a,b]>=8)
hard_negative=same & ((context[a,b]<np.maximum(threshold-.12,0)) | (lift[a,b]>1.5)) & ~positive
cross=~same
rng=np.random.default_rng(SEED)
pos_ids=np.flatnonzero(positive)
if len(pos_ids)<30:
    raise ValueError('Too few weak positive pairs. Increase catalogue coverage or inspect category/context thresholds.')
neg_ids=set(np.flatnonzero(hard_negative).tolist())
for mask in [cross,complement & ~positive]:
    pool=np.flatnonzero(mask)
    neg_ids.update(rng.choice(pool,min(len(pool),len(pos_ids)*2),replace=False).tolist())
chosen=np.concatenate([pos_ids,np.array(sorted(neg_ids),dtype=int)])
labelled=pair_table.iloc[chosen].copy().reset_index(drop=True)
labelled['label']=positive[chosen].astype(int)
labelled['pair_group']=labelled.a.astype(str)+'_'+labelled.b.astype(str)
X=pair_features(labelled.a,labelled.b); y=labelled.label.to_numpy()
# Product-disjoint outer split: discard pairs spanning the boundary.
product_rng=np.random.default_rng(SEED)
test_products=set()
for category in np.unique(cat):
    ids=np.flatnonzero(cat==category); product_rng.shuffle(ids)
    test_products.update(ids[:max(2,round(len(ids)*.25))].tolist())
train_products=set(range(len(products)))-test_products
pa=labelled.a.to_numpy(); pb=labelled.b.to_numpy()
train_idx=np.flatnonzero(np.isin(pa,list(train_products)) & np.isin(pb,list(train_products)))
test_idx=np.flatnonzero(np.isin(pa,list(test_products)) & np.isin(pb,list(test_products)))
def endpoints(indices):
    return set(pa[indices]) | set(pb[indices])
assert not endpoints(train_idx) & endpoints(test_idx)
# Purge cross-boundary pairs in every fold; both endpoints must be disjoint.
fold_of={}
for category in np.unique(cat):
    ids=np.array(sorted(train_products & set(np.flatnonzero(cat==category))))
    product_rng.shuffle(ids)
    fold_of.update({int(pid):j%3 for j,pid in enumerate(ids)})
fa=np.array([fold_of[int(v)] for v in pa[train_idx]])
fb=np.array([fold_of[int(v)] for v in pb[train_idx]])
cv=[(np.flatnonzero((fa!=f)&(fb!=f)),np.flatnonzero((fa==f)&(fb==f))) for f in range(3)]
for tr,va in cv:
    assert not endpoints(train_idx[tr]) & endpoints(train_idx[va])
    assert len(np.unique(y[train_idx[tr]]))==len(np.unique(y[train_idx[va]]))==2
print(f'{len(train_idx):,} training / {len(test_idx):,} product-disjoint test pairs; {len(labelled)-len(train_idx)-len(test_idx):,} crossing pairs purged.')
print('All six models share three product-disjoint CV folds. Historical context is transductive; labels are weak heuristics.')


4,133 training / 443 product-disjoint test pairs; 2,861 crossing pairs purged.
All six models share three product-disjoint CV folds. Historical context is transductive; labels are weak heuristics.


### 6 — Two proposed models, four comparators, product-disjoint evaluation
RF and GB are proposed; LR, XGBoost, MLP and soft voting are comparisons. Every model uses identical product-disjoint train/test indices and three purged product-disjoint CV folds. Pairs spanning a boundary are excluded. Sigmoid calibration averages fold-specific calibrated estimators (`ensemble=True`), so each calibrator sees held-out products. Deployment uses training-CV average precision only. Scaling stays inside LR/MLP pipelines. Tuning is brief and non-nested; CV selection estimates can be optimistic.

Historical basket-derived context and label thresholds use the available catalogue history (transductive). Product-disjoint supervised labels do not imply a fully inductive cold-start or expert-labelled benchmark. The separate next-order benchmark measures observed switches, not causal recovery. [Calibration reference](https://scikit-learn.org/stable/modules/generated/sklearn.calibration.CalibratedClassifierCV.html).


In [7]:
from importlib.metadata import version as package_version

MODEL_ORDER=['Random Forest','Gradient Boosting','Logistic Regression','XGBoost','MLPClassifier','Soft Voting (RF + GB + XGB)']
PROPOSED=['Random Forest','Gradient Boosting']
MODEL_ROLES={name:('Proposed' if name in PROPOSED else 'Comparison') for name in MODEL_ORDER}
recipe_version='product-disjoint-v3'
model_hash=hashlib.sha256(X.tobytes()+y.tobytes()+train_idx.tobytes()+test_idx.tobytes()+
    recipe_version.encode()+package_version('scikit-learn').encode()+package_version('xgboost').encode()).hexdigest()[:16]
model_file=CACHE/f'models-{model_hash}.joblib'
model_cache_hit=model_file.exists()
if model_cache_hit:
    models,cv_results=joblib.load(model_file)
    print('Loaded six cached fitted models; training times are the original measured times.')
else:
    specs={
      'Random Forest':(RandomForestClassifier(random_state=SEED,n_jobs=1,class_weight='balanced'),
          {'n_estimators':[140],'max_depth':[7,None],'min_samples_leaf':[3,8]}),
      'Gradient Boosting':(GradientBoostingClassifier(random_state=SEED),
          {'n_estimators':[90,150],'learning_rate':[.06],'max_depth':[2,3]}),
      'Logistic Regression':(Pipeline([('scale',StandardScaler()),('classifier',LogisticRegression(max_iter=1000,class_weight='balanced',random_state=SEED))]),
          {'classifier__C':[.3,3.]}),
      'XGBoost':(XGBClassifier(objective='binary:logistic',eval_metric='logloss',tree_method='hist',
          random_state=SEED,n_jobs=1,learning_rate=.06,subsample=.9,colsample_bytree=.9),
          {'n_estimators':[100,160],'max_depth':[2,3]}),
      'MLPClassifier':(Pipeline([('scale',StandardScaler()),('classifier',MLPClassifier(
          random_state=SEED,max_iter=300,early_stopping=True,validation_fraction=.15,n_iter_no_change=18))]),
          {'classifier__hidden_layer_sizes':[(32,),(64,)],'classifier__alpha':[.001]})}
    models={}; cv_results=[]; best_estimators={}
    def tune_and_calibrate(name,estimator,grid):
        t=perf_counter()
        # The exact same `cv` index lists are passed to every search and calibration.
        with threadpool_limits(limits=1), joblib.parallel_backend('threading',n_jobs=2):
            search=GridSearchCV(estimator,grid,scoring='average_precision',cv=cv,n_jobs=2,error_score='raise')
            search.fit(X[train_idx],y[train_idx])
            calibrated=CalibratedClassifierCV(estimator=search.best_estimator_,method='sigmoid',cv=cv,ensemble=True,n_jobs=2)
            calibrated.fit(X[train_idx],y[train_idx])
        models[name]=calibrated
        best_estimators[name]=clone(search.best_estimator_)
        cv_results.append(dict(model=name,role=MODEL_ROLES[name],cv_pr_auc=float(search.best_score_),
            cv_std=float(search.cv_results_['std_test_score'][search.best_index_]),
            configurations=len(search.cv_results_['params']),parameters=search.best_params_,fit_seconds=perf_counter()-t))
        print(f'{name}: CV PR-AUC {search.best_score_:.6f}, {cv_results[-1]["fit_seconds"]:.2f}s')
    for name,(estimator,grid) in specs.items():
        tune_and_calibrate(name,estimator,grid)
    voter=VotingClassifier(estimators=[('rf',best_estimators['Random Forest']),('gb',best_estimators['Gradient Boosting']),
        ('xgb',best_estimators['XGBoost'])],voting='soft',n_jobs=1)
    tune_and_calibrate(MODEL_ORDER[-1],voter,{'weights':[(1,1,1),(1,1,2)]})
    joblib.dump((models,cv_results),model_file)
assert list(models)==MODEL_ORDER
for result in cv_results:
    result['cv_pr_auc']=float(np.clip(result['cv_pr_auc'],0,1))
# No outer test / future-order metric is read when choosing the deployed model.
selected_name=max(cv_results,key=lambda row:round(row['cv_pr_auc'],12))['model']
model=models[selected_name]
model_metrics=[]; curves={}; cms={}
for name,est in models.items():
    prob=est.predict_proba(X[test_idx])[:,1]; pred=prob>=.5
    timing=next(row for row in cv_results if row['model']==name)
    model_metrics.append(dict(model=name,role=MODEL_ROLES[name],roc_auc=roc_auc_score(y[test_idx],prob),
       pr_auc=average_precision_score(y[test_idx],prob),f1=f1_score(y[test_idx],pred),
       precision=precision_score(y[test_idx],pred,zero_division=0),recall=recall_score(y[test_idx],pred,zero_division=0),
       brier=brier_score_loss(y[test_idx],prob),cv_pr_auc=timing['cv_pr_auc'],fit_seconds=timing['fit_seconds']))
    fpr,tpr,_=roc_curve(y[test_idx],prob)
    prec,rec,_=precision_recall_curve(y[test_idx],prob)
    curves[name]={'fpr':fpr.tolist(),'tpr':tpr.tolist(),'precision':prec.tolist(),'recall':rec.tolist()}
    cms[name]=confusion_matrix(y[test_idx],pred,labels=[0,1]).tolist()
print('Best / deployed by training CV:',selected_name)
print('All six models use',len(train_idx),'training pairs,',len(test_idx),'test pairs and the same',len(cv),'CV folds.')
# Model-agnostic importance supports LR, MLP and the voter as well as trees.
# This is a post-selection diagnostic; it never feeds model selection or training.
with threadpool_limits(limits=1), joblib.parallel_backend('threading',n_jobs=2):
    perm=permutation_importance(model,X[test_idx],y[test_idx],scoring='average_precision',n_repeats=3,random_state=SEED,n_jobs=2)
importance=pd.DataFrame({'feature':FEATURES,'importance':perm.importances_mean,'std':perm.importances_std}).sort_values('importance')
px.bar(importance,x='importance',y='feature',orientation='h',title='Deployed model · permutation feature importance',labels={'importance':'Decrease in test PR-AUC','feature':'Feature'}).show()
px.imshow(cms[selected_name],text_auto=True,x=['Predicted non-substitute','Predicted substitute'],y=['Actual non-substitute','Actual substitute'],title='Untouched weak-label test pairs · deployed confusion matrix',color_continuous_scale='Greens').show()
for kind in ['ROC','PR']:
    fig=go.Figure()
    for name,cur in curves.items():
        fig.add_trace(go.Scatter(x=cur['fpr' if kind=='ROC' else 'recall'],y=cur['tpr' if kind=='ROC' else 'precision'],name=name,mode='lines'))
    fig.update_layout(title=f'{kind} curves · identical weak-label test pairs',xaxis_title='False positive rate' if kind=='ROC' else 'Recall',yaxis_title='True positive rate' if kind=='ROC' else 'Precision')
    fig.show()


Loaded six cached fitted models; training times are the original measured times.
Best / deployed by training CV: Random Forest
All six models use 4133 training pairs, 443 test pairs and the same 3 CV folds.


### 7 · One six-model comparison table and chart, plus future-order baselines
Candidate alternatives share the same category and have lift below 1.5; this excludes strong complement edges. No held-out outcomes tune this eligibility policy. All eligible products compete; the future purchased target is never inserted into candidates. Scores are calibrated weak-label probabilities, **not a probability of customer acceptance**.

For each household, compare its penultimate basket with its held-out final basket. A removed product is eligible as a query when a different same-aisle product appears next and the original does not. On synthetic data only, the target must also share the hidden need group. The relevant set contains all observed qualifying next-order items, not an inferred heuristic label. Metrics average over household-query switching events with at least one observed target; confidence intervals bootstrap households. This is a conditional switching benchmark and excludes no-switch orders; its event count and target eligibility coverage are reported. It is not a causal stockout experiment.

Precision@5 = hits / 5; Recall@5 = hits / relevant targets; NDCG discounts lower positions; MAP averages precision at relevant ranks over all relevant targets. Baselines use exactly the same eligible pool. Popularity orders same-category products first; content ranks TF-IDF name cosine. Revenue uses the actual future target price, counted once per removed product.

Each of the six calibrated models ranks the **same eligible products for the same future switching events**. The combined comparison table joins pair-test ROC-AUC, PR-AUC and F1 to independently measured Precision@5, Recall@5, NDCG@5, MAP@5 and original training time. The green row / starred bar is the deployment winner **by training CV**, not a claim that it wins every held-out metric. The single interactive bar chart provides a metric dropdown, so training seconds are never mixed with unitless quality scores on one axis. MLP's early-stopping validation is internal to each training fold. All other methods use the same outer folds.


In [8]:
n=len(products)
aa,bb=np.indices((n,n))
all_pair_features=pair_features(aa.ravel(),bb.ravel())
eligible=(cat[:,None]==cat[None,:]) & (lift<1.5)
np.fill_diagonal(eligible,False)
model_scores={}
with threadpool_limits(limits=1):
    for name,est in models.items():
        raw=est.predict_proba(all_pair_features)[:,1].reshape(n,n)
        values=np.where(eligible,raw,-1).astype(np.float32)
        np.fill_diagonal(values,-1)
        model_scores[name]=values
score=model_scores[selected_name]
past=history.merge(history_orders[['order_id','user_id','order_number']],on='order_id')
penultimate=history_orders.loc[history_orders.groupby('user_id').order_number.idxmax()]
past_sets={u:set(g.product_id.map(pid_to_idx)) for u,g in past[past.order_id.isin(penultimate.order_id)].groupby('user_id')}
next_sets={u:set(g.product_id.map(pid_to_idx)) for u,g in future.merge(future_orders[['order_id','user_id']],on='order_id').groupby('user_id')}
events=[]
for user,previous in past_sets.items():
    nxt=next_sets.get(user,set())
    for q in sorted(previous-nxt):
        relevant={t for t in nxt if aisle[t]==aisle[q] and (source=='Instacart' or products.iloc[t].need_id==products.iloc[q].need_id)}
        if relevant:
            events.append({'user':int(user),'query':int(q),'targets':sorted(relevant)})
if not events:
    raise ValueError('No future switching events found. Increase household sample / catalogue coverage.')

K=5
def evaluate_ranker(values):
    lookup={q:np.flatnonzero(eligible[q])[np.argsort(-values[q,eligible[q]],kind='stable')][:K].tolist() for q in {e['query'] for e in events}}
    records=[]
    for e in events:
        top=lookup[e['query']]; targets=set(e['targets'])
        rel=np.array([int(t in targets) for t in top])
        hits=int(rel.sum())
        dcg=float(np.sum(rel/np.log2(np.arange(len(rel))+2)))
        ideal=float(np.sum(1/np.log2(np.arange(min(K,len(targets)))+2)))
        ap=float(np.sum((np.cumsum(rel)/(np.arange(len(rel))+1))*rel)/len(targets))
        recovered=[t for t in top if t in targets]
        records.append(dict(user=e['user'],query=e['query'],precision_at_k=hits/K,recall_at_k=hits/len(targets),
            ndcg_at_k=dcg/ideal,map_at_k=ap,hit=int(hits>0),revenue=float(prices[recovered[0]]) if recovered else 0.,
            eligible_target=int(any(eligible[e['query'],t] for t in targets))))
    frame=pd.DataFrame(records)
    agg=frame.groupby('user').agg(hits=('hit','sum'),n=('hit','size'))
    rng=np.random.default_rng(SEED)
    draws=rng.integers(0,len(agg),size=(400,len(agg)))
    boot=agg.hits.to_numpy()[draws].sum(1)/agg.n.to_numpy()[draws].sum(1)
    summary=dict(**frame[['precision_at_k','recall_at_k','ndcg_at_k','map_at_k']].mean().to_dict(),
        hit_rate=float(frame.hit.mean()),hit_ci_low=float(np.quantile(boot,.025)),hit_ci_high=float(np.quantile(boot,.975)),
        target_coverage=float(frame.eligible_target.mean()),events=len(frame),revenue_saved=float(frame.revenue.sum()))
    return summary,frame

model_ranking_rows=[]; event_details=[]
for name,values in model_scores.items():
    result,details=evaluate_ranker(values)
    model_ranking_rows.append(dict(model=name,**result))
    if name==selected_name: event_details=details.to_dict('records')
model_ranking_metrics=pd.DataFrame(model_ranking_rows)
pair_metrics=pd.DataFrame(model_metrics)
joined=pair_metrics.merge(model_ranking_metrics,on='model',validate='one_to_one')
joined['deployed']=joined.model.eq(selected_name)
comparison=joined[['model','role','deployed','roc_auc','pr_auc','f1','precision_at_k','recall_at_k','ndcg_at_k','map_at_k','fit_seconds','cv_pr_auc']].rename(columns={
    'model':'Model','role':'Role','deployed':'Deployed','roc_auc':'ROC-AUC','pr_auc':'PR-AUC','f1':'F1',
    'precision_at_k':'Precision@5','recall_at_k':'Recall@5','ndcg_at_k':'NDCG@5','map_at_k':'MAP@5','fit_seconds':'Training time (s)','cv_pr_auc':'CV PR-AUC'})
comparison['Model']=pd.Categorical(comparison['Model'],categories=MODEL_ORDER,ordered=True)
comparison=comparison.sort_values('Model').reset_index(drop=True)
comparison['Model']=comparison['Model'].astype(str)
def highlight_deployed(row):
    return ['background-color:#EAF4EE;color:#084D39;font-weight:600' if row['Model']==selected_name else '' for _ in row]
display(comparison.style.apply(highlight_deployed,axis=1).format({c:('{:.2f}' if c=='Training time (s)' else '{:.6f}') for c in comparison.select_dtypes('number').columns}))
def make_comparison_figure(frame,selected):
    # One interactive bar chart: the dropdown keeps unlike units off the same axis.
    metrics=['PR-AUC','ROC-AUC','F1','Precision@5','Recall@5','NDCG@5','MAP@5','Training time (s)','CV PR-AUC']
    labels=[('★ ' if name==selected else '')+('Soft Voting' if name=='Soft Voting (RF + GB + XGB)' else name) for name in frame['Model']]
    colors=['#0C6B4F' if name==selected else ('#4B7C69' if role=='Proposed' else '#5C7293')
            for name,role in zip(frame['Model'],frame['Role'])]
    fig=go.Figure()
    for i,metric in enumerate(metrics):
        values=frame[metric].astype(float).to_numpy()
        text=[f'{v:.2f}s' if metric=='Training time (s)' else f'{v:.4f}' for v in values]
        fig.add_trace(go.Bar(x=values,y=labels,orientation='h',name=metric,visible=i==0,
            marker_color=colors,text=text,textposition='outside',cliponaxis=False,customdata=frame['Model'].tolist(),
            hovertemplate='%{customdata}<br>'+metric+': %{x:.6f}<extra></extra>'))
    buttons=[]
    for i,metric in enumerate(metrics):
        maximum=float(frame[metric].max())
        buttons.append(dict(label=metric,method='update',args=[{'visible':[j==i for j in range(len(metrics))]},
            {'xaxis.title.text':metric,'xaxis.range':[0,max(1.15,maximum*1.22)],'title.text':'Model comparison · '+metric}]))
    fig.update_layout(title=dict(text='Model comparison · PR-AUC',x=0,y=.98,yanchor='top'),height=460,showlegend=False,meta={'comparison':True},
        font=dict(family='Inter',color='#4B5A52'),barcornerradius=5,
        xaxis=dict(title='PR-AUC',range=[0,1.15],showgrid=False,zeroline=False),
        yaxis=dict(autorange='reversed',automargin=True,showgrid=False),margin=dict(l=12,r=84,t=140,b=48),
        updatemenus=[dict(buttons=buttons,direction='down',x=1,y=1.15,xanchor='right',yanchor='top',
            bgcolor='white',bordercolor='#CFE3D8',font=dict(color='#193D2D'))],
        paper_bgcolor='rgba(0,0,0,0)',plot_bgcolor='rgba(0,0,0,0)')
    return fig

comparison_figure=make_comparison_figure(comparison,selected_name)
comparison_figure.show()
print('Chart dropdown: ROC-AUC, PR-AUC, F1, Precision@5, Recall@5, NDCG@5, MAP@5, training seconds and training CV PR-AUC.')

rankings={'FreshMatch':score,
    'Popular in category':np.where(eligible,(cat[:,None]==cat[None,:])*2+(pop/(pop.max()+1))[None,:],-1),
    'Content similarity':np.where(eligible,text_sim,-1)}
ranking_rows=[]
for name,values in rankings.items():
    result,_=evaluate_ranker(values)
    ranking_rows.append(dict(model=name,**result))
ranking_metrics=pd.DataFrame(ranking_rows)
display(ranking_metrics.round(4))
px.bar(ranking_metrics.melt(id_vars='model',value_vars=['precision_at_k','recall_at_k','ndcg_at_k','map_at_k']),x='variable',y='value',color='model',barmode='group',title='Deployed model vs content / popularity · same next-order candidate pools',labels={'variable':'Metric @ 5','value':'Score'}).show()
best_row=ranking_metrics[ranking_metrics.model=='FreshMatch'].iloc[0]
print(f'Deployed model: {selected_name}. Conditional top-5 switch hit rate: {best_row.hit_rate:.1%} across {len(events):,} events.')
print(f'95% household-bootstrap interval: {best_row.hit_ci_low:.1%}–{best_row.hit_ci_high:.1%}.')
print('Offline simulated opportunities, not verified recovered sales. Future-order results do not choose the model.')

# Local, model-based single-feature occlusion against training medians.
# These are sensitivity deltas, not additive SHAP values or causal effects.
local_effects=np.zeros((n,n,len(FEATURES)),dtype=np.float32)
valid_flat=np.flatnonzero(eligible.ravel())
feature_reference=np.median(X[train_idx],axis=0)
base_probability=model.predict_proba(all_pair_features[valid_flat])[:,1]
with threadpool_limits(limits=1):
    for j in range(len(FEATURES)):
        altered=all_pair_features[valid_flat].copy(); altered[:,j]=feature_reference[j]
        local_effects.reshape(-1,len(FEATURES))[valid_flat,j]=base_probability-model.predict_proba(altered)[:,1]
print('Saved local feature sensitivities for every eligible recommendation.')


,Model,Role,Deployed,ROC-AUC,PR-AUC,F1,Precision@5,Recall@5,NDCG@5,MAP@5,Training time (s),CV PR-AUC
0,Random Forest,Proposed,True,1.000000,1.000000,0.992806,0.105385,0.526925,0.323789,0.257733,8.65,1.000000
1,Gradient Boosting,Proposed,False,0.998506,0.994889,0.985507,0.098873,0.494364,0.292005,0.226680,6.30,0.999780
2,Logistic Regression,Comparison,False,1.000000,1.000000,1.000000,0.121165,0.605823,0.377826,0.303423,0.16,1.000000
3,XGBoost,Comparison,False,0.997664,0.993290,0.985507,0.105949,0.529743,0.324162,0.257561,2.27,1.000000
4,MLPClassifier,Comparison,False,1.000000,1.000000,1.000000,0.109017,0.545085,0.343583,0.277593,4.04,0.999652
5,Soft Voting (RF + GB + XGB),Comparison,False,1.000000,1.000000,0.985507,0.106637,0.533187,0.324853,0.257034,7.94,1.000000


Chart dropdown: ROC-AUC, PR-AUC, F1, Precision@5, Recall@5, NDCG@5, MAP@5, training seconds and training CV PR-AUC.


,model,precision_at_k,recall_at_k,ndcg_at_k,map_at_k,hit_rate,hit_ci_low,hit_ci_high,target_coverage,events,revenue_saved
0,FreshMatch,0.1054,0.5269,0.3238,0.2577,0.5269,0.5102,0.5442,1.0,3194,6859.76
1,Popular in category,0.0643,0.3215,0.1957,0.1547,0.3215,0.3055,0.3356,1.0,3194,3522.38
2,Content similarity,0.1035,0.5175,0.3418,0.2841,0.5175,0.5012,0.5358,1.0,3194,6825.53


Deployed model: Random Forest. Conditional top-5 switch hit rate: 52.7% across 3,194 events.
95% household-bootstrap interval: 51.0%–54.4%.
Offline simulated opportunities, not verified recovered sales. Future-order results do not choose the model.


Saved local feature sensitivities for every eligible recommendation.


### 8 · Save versioned artifacts and a measured results summary
The inference bundle contains the calibrated model, TF-IDF vocabulary, feature names and precomputed score/context/lift matrices. Queries are matrix lookups, so repeated recommendations avoid model inference. Joblib files are trusted local outputs; never load unknown pickle/joblib files. A manifest records the data source, limitations and hashes. Generated source has no hard-coded evaluation results.

In [9]:
products.to_csv(ART/'products.csv',index=False)
rules.to_csv(ART/'rules.csv',index=False)
importance.to_csv(ART/'importance.csv',index=False)
pd.DataFrame(model_metrics).to_csv(ART/'model_metrics.csv',index=False)
comparison.to_csv(ART/'comparison.csv',index=False)
model_ranking_metrics.to_csv(ART/'model_ranking_metrics.csv',index=False)
labelled.to_csv(ART/'labelled_pairs.csv',index=False)
split_payload={'train_idx':train_idx,'test_idx':test_idx}
for i,(fold_train,fold_valid) in enumerate(cv):
    split_payload[f'cv_train_{i}']=fold_train; split_payload[f'cv_valid_{i}']=fold_valid
np.savez_compressed(ART/'evaluation_split.npz',**split_payload)
(ART/'comparison_chart.json').write_text(comparison_figure.to_json(),encoding='utf-8')
ranking_metrics.to_csv(ART/'ranking_metrics.csv',index=False)
mining_compare.to_csv(ART/'mining_compare.csv',index=False)
pd.DataFrame(event_details).to_csv(ART/'switch_events.csv',index=False)
np.savez_compressed(ART/'matrices.npz',scores=score,context=context,lift=lift,text=text_sim,eligible=eligible,local_effects=local_effects,feature_names=np.array(FEATURES))
joblib.dump({'models':models,'selected':selected_name,'features':FEATURES,'tfidf':tfidf},ART/'models.joblib')
(ART/'curves.json').write_text(json.dumps({'curves':curves,'confusion':cms}),encoding='utf-8')
(ART/'cv_results.json').write_text(json.dumps(cv_results,indent=2),encoding='utf-8')
manifest=dict(version='3.0',seed=SEED,source=source,fallback_reason=fallback_reason,
    model_cache_hit=model_cache_hit,software_versions={name:package_version(name) for name in ['numpy','pandas','scikit-learn','xgboost','mlxtend']},
    products=n,history_baskets=len(history_orders),future_baskets=len(future_orders),events=len(events),
    selected_model=selected_name,proposed_models=PROPOSED,comparison_models=[name for name in MODEL_ORDER if name not in PROPOSED],
    selection_criterion='Highest training-CV average precision; held-out metrics are reporting only',
    training_time_definition='Grid search + refit + sigmoid calibration, including component fits for the voter; original timings retained on cache hits',
    importance_method='Permutation decrease in held-out PR-AUC, 3 repeats; post-selection diagnostic',price_note='Synthetic USD prices' if source!='Instacart' else 'Estimated USD prices; not supplied by Instacart',
    brand_note='Simulated brands' if source!='Instacart' else 'First-name-token proxy, not verified brands',
    support=MIN_SUPPORT,weak_pairs=len(labelled),test_pairs=len(test_idx),
    elapsed_seconds=round(perf_counter()-START,2),
    limitations=['Weak labels are heuristic; test accuracy is not expert validation.',
       'Both product endpoints are disjoint across supervised splits; historical context remains shared (transductive).',
       'Next-order benchmark is conditional on observed switches, not all stockouts.',
       'Scores estimate weak-label suitability, not customer acceptance or dietary safety.',
       'No causal claim of recovered revenue; prices are synthetic or estimated.'])
manifest['sha256']={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in ART.iterdir() if p.is_file() and p.name!='manifest.json'}
(ART/'manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
def md_table(frame):
    rows=['| '+' | '.join(map(str,frame.columns))+' |','| '+' | '.join(['---']*len(frame.columns))+' |']
    rows += ['| '+' | '.join(str(round(v,4)) if isinstance(v,(float,np.floating)) else str(v) for v in row)+' |' for row in frame.itertuples(index=False,name=None)]
    return '\n'.join(rows)
summary=f"""# FreshMatch · Measured results
Data: **{source}**. {n} products; {len(history_orders):,} history baskets; {len(future_orders):,} future baskets.
Selected model: **{selected_name}**, chosen by training cross-validation.

## Six-model comparison · identical pair split, CV folds and next-order events
Proposed methods: **Random Forest and Gradient Boosting**. Comparison methods: Logistic Regression, XGBoost, MLPClassifier and soft voting (RF + GB + XGBoost).

{md_table(comparison)}

The deployed row is selected by **training CV PR-AUC**, not by these held-out outcomes. The notebook and Model Performance page share this exact table and a single bar chart with a metric dropdown (including training seconds on their own scale).
Training time includes grid search, refit and calibration; voting time includes component fits. LR/MLP scaling is fitted within each training fold. Exact CV ties retain the declared model order.


## Held-out next-order ranking (K=5)
{md_table(ranking_metrics)}

Classification PR-AUC is average precision. MAP is truncated at 5 and divided by all relevant targets.
Top-5 hit rate has a household-bootstrap 95% interval of {best_row.hit_ci_low:.1%}–{best_row.hit_ci_high:.1%}.
Runtime before app launch: {manifest['elapsed_seconds']:.1f} seconds (excludes package installation; fitted-model cache hit: {model_cache_hit}). The training-time column retains original search/refit/calibration measurements when models are cached.

Novelty: complementary-item rules are separated from substitution signals; context similarity measures shared shopping roles;
low direct lift suppresses co-purchased products; six calibrated model families share one split/CV protocol and are compared with content and popularity;
the benchmark uses future behavior independent of weak labels; multi-item stockouts remove unavailable candidates.

Limitations: synthetic/estimated prices, heuristic labels, product-disjoint supervised split with transductive historical context, and a conditional observational
switch benchmark. Results are not causal sales recovery evidence. A simpler baseline can outperform the ensemble;
the tables report measured outcomes without hiding that possibility.
"""
(ROOT/'RESULTS.md').write_text(summary,encoding='utf-8')
display(Markdown(summary))
print('Artifacts saved:',ART)

# FreshMatch · Measured results
Data: **Synthetic grocery simulation**. 288 products; 24,000 history baskets; 3,000 future baskets.
Selected model: **Random Forest**, chosen by training cross-validation.

## Six-model comparison · identical pair split, CV folds and next-order events
Proposed methods: **Random Forest and Gradient Boosting**. Comparison methods: Logistic Regression, XGBoost, MLPClassifier and soft voting (RF + GB + XGBoost).

| Model | Role | Deployed | ROC-AUC | PR-AUC | F1 | Precision@5 | Recall@5 | NDCG@5 | MAP@5 | Training time (s) | CV PR-AUC |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Random Forest | Proposed | True | 1.0 | 1.0 | 0.9928 | 0.1054 | 0.5269 | 0.3238 | 0.2577 | 8.6515 | 1.0 |
| Gradient Boosting | Proposed | False | 0.9985 | 0.9949 | 0.9855 | 0.0989 | 0.4944 | 0.292 | 0.2267 | 6.3029 | 0.9998 |
| Logistic Regression | Comparison | False | 1.0 | 1.0 | 1.0 | 0.1212 | 0.6058 | 0.3778 | 0.3034 | 0.1559 | 1.0 |
| XGBoost | Comparison | False | 0.9977 | 0.9933 | 0.9855 | 0.1059 | 0.5297 | 0.3242 | 0.2576 | 2.2667 | 1.0 |
| MLPClassifier | Comparison | False | 1.0 | 1.0 | 1.0 | 0.109 | 0.5451 | 0.3436 | 0.2776 | 4.044 | 0.9997 |
| Soft Voting (RF + GB + XGB) | Comparison | False | 1.0 | 1.0 | 0.9855 | 0.1066 | 0.5332 | 0.3249 | 0.257 | 7.9429 | 1.0 |

The deployed row is selected by **training CV PR-AUC**, not by these held-out outcomes. The notebook and Model Performance page share this exact table and a single bar chart with a metric dropdown (including training seconds on their own scale).
Training time includes grid search, refit and calibration; voting time includes component fits. LR/MLP scaling is fitted within each training fold. Exact CV ties retain the declared model order.


## Held-out next-order ranking (K=5)
| model | precision_at_k | recall_at_k | ndcg_at_k | map_at_k | hit_rate | hit_ci_low | hit_ci_high | target_coverage | events | revenue_saved |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| FreshMatch | 0.1054 | 0.5269 | 0.3238 | 0.2577 | 0.5269 | 0.5102 | 0.5442 | 1.0 | 3194 | 6859.76 |
| Popular in category | 0.0643 | 0.3215 | 0.1957 | 0.1547 | 0.3215 | 0.3055 | 0.3356 | 1.0 | 3194 | 3522.38 |
| Content similarity | 0.1035 | 0.5175 | 0.3418 | 0.2841 | 0.5175 | 0.5012 | 0.5358 | 1.0 | 3194 | 6825.53 |

Classification PR-AUC is average precision. MAP is truncated at 5 and divided by all relevant targets.
Top-5 hit rate has a household-bootstrap 95% interval of 51.0%–54.4%.
Runtime before app launch: 21.0 seconds (excludes package installation; fitted-model cache hit: True). The training-time column retains original search/refit/calibration measurements when models are cached.

Novelty: complementary-item rules are separated from substitution signals; context similarity measures shared shopping roles;
low direct lift suppresses co-purchased products; six calibrated model families share one split/CV protocol and are compared with content and popularity;
the benchmark uses future behavior independent of weak labels; multi-item stockouts remove unavailable candidates.

Limitations: synthetic/estimated prices, heuristic labels, product-disjoint supervised split with transductive historical context, and a conditional observational
switch benchmark. Results are not causal sales recovery evidence. A simpler baseline can outperform the ensemble;
the tables report measured outcomes without hiding that possibility.


Artifacts saved: C:\AML\artifacts


### 9 · Write the complete Streamlit application
This cell creates `app.py` as a runtime artifact with `%%writefile`, as requested. All six pages load the saved outputs. The home page uses Streamlit's searchable dropdown for autocomplete, bounded filters, cached inference, plain-language reasons and CSV export. Multi-item stockouts are enforced across every candidate pool. Simulator KPIs explicitly apply the measured conditional switch-hit rate and are labelled scenario estimates. No model score is represented as an acceptance probability.

In [10]:
%%writefile app.py
from pathlib import Path
import json, html, time, textwrap
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import streamlit as st

st.set_page_config(page_title='FreshMatch · Better alternatives',page_icon='🌿',layout='wide',initial_sidebar_state='auto')
BASE=Path(__file__).resolve().parent
ART=BASE/'artifacts'
ACCENT='#0F7B5F'
px.defaults.template='plotly_white'
px.defaults.color_discrete_sequence=[ACCENT,'#99620B','#3E65A0','#AF4035']
st.markdown("""<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap');
/* FreshMatch design tokens: an 8 px spacing scale, AA text colours, one brand green. */
:root{--primary:#0F7B5F;--primary-hover:#084D39;--ink:#193D2D;--muted:#4B5A52;--background:#F6FAF7;--card:#FFFFFF;--border:#CFE3D8;--soft:#EAF4EE;--amber:#875608;--amber-bg:#FFF3D9;--danger:#A33226;--danger-bg:#FFF0EC;--space-1:8px;--space-2:16px;--space-3:24px;--space-4:32px;--radius:16px;--shadow:0 4px 20px rgba(25,61,45,.045)}
html,body,.stApp,[data-testid="stAppViewContainer"]{background:var(--background);color:var(--ink);font-family:'Inter',sans-serif;font-weight:400}
/* Remove Streamlit chrome, retaining only the essential mobile sidebar opener. */
[data-testid="stHeader"]{height:0!important;background:transparent!important;visibility:hidden;pointer-events:none}
#MainMenu,footer,[data-testid="stDecoration"],[data-testid="stStatusWidget"]{display:none!important}
[data-testid="stToolbar"]{visibility:hidden;pointer-events:none}
[data-testid="stSidebarCollapsedControl"]{visibility:visible!important;pointer-events:auto;position:fixed!important;left:8px!important;top:8px!important;z-index:1000}
[data-testid="stExpandSidebarButton"]{visibility:visible!important;pointer-events:auto!important;position:fixed!important;top:8px!important;left:8px!important;z-index:1000;background:var(--card)!important;color:var(--primary)!important;border:1px solid var(--border)!important;border-radius:10px!important;min-width:32px;min-height:32px}
[data-testid="stSidebarCollapsedControl"] button{background:var(--card)!important;color:var(--primary)!important;border:1px solid var(--border)!important;border-radius:10px}
.block-container{padding:2.5rem 2.5rem 2rem;max-width:1344px;min-width:0}
[data-testid="stMain"] [data-testid="stVerticalBlock"]{gap:12px}
h1,h2,h3{font-family:'Inter',sans-serif!important;letter-spacing:-.035em!important;color:var(--ink)!important;overflow-wrap:anywhere}
h1{font-size:2rem!important;font-weight:700!important;line-height:1.2!important;margin:0!important;padding:0!important}
h2{font-size:1.45rem!important;font-weight:700!important}h3{font-size:1.05rem!important;font-weight:600!important}
.stApp p,.stApp li{font-weight:400;line-height:1.6}
[data-testid="stCaptionContainer"], [data-testid="stCaptionContainer"] p{color:var(--muted)!important;font-weight:400!important;opacity:1!important;font-size:.78rem}
[data-testid="stWidgetLabel"], [data-testid="stWidgetLabel"] p, label, [data-testid="stSlider"] [data-testid="stMarkdownContainer"]{color:var(--ink)!important;font-weight:600!important;opacity:1!important}
[data-testid="stSlider"] [role="slider"], [data-testid="stSliderTickBar"]{color:var(--muted)!important;font-weight:600!important}
.eyebrow{color:var(--primary);font-size:.66rem;font-weight:700;line-height:1.6;letter-spacing:.15em;text-transform:uppercase;margin:0 0 8px;overflow:visible}
.hero{background:linear-gradient(115deg,#E8F3EB,#F0F6E8);border:1px solid var(--border);border-radius:var(--radius);padding:20px 24px;margin:0;min-height:100px}
.hero h1{font-size:2rem!important;line-height:1.2!important;max-width:100%}
.hero p{font-size:.875rem;color:var(--muted);font-weight:400;line-height:1.45;margin:8px 0 0}
[data-testid="stSidebar"]{background:#EDF5EF!important;border-right:1px solid var(--border);min-width:0}
[data-testid="stSidebar"] [data-testid="stSidebarUserContent"]{padding:24px 16px!important}
[data-testid="stSidebar"] .block-container{padding:16px!important}
.brand{display:flex;align-items:center;gap:10px;font-size:1.5rem;font-weight:800;letter-spacing:-.055em;color:var(--ink);line-height:1.2}
.brand-icon{font-size:32px;line-height:1}.brand-name span{color:var(--primary)}
.subbrand{font-size:.65rem;letter-spacing:.1em;color:var(--muted);font-weight:600;margin:8px 0 24px 42px}
.nav-label{font-size:.65rem;font-weight:700;color:var(--muted);letter-spacing:.12em;margin:0 0 8px;text-transform:uppercase}
[data-testid="stSidebar"] .stButton>button{min-height:44px;width:100%;justify-content:flex-start;text-align:left;padding:10px 16px;border:1px solid transparent!important;border-radius:24px!important;background:transparent;color:var(--ink);font-weight:600;box-shadow:none;transition:background .18s}
[data-testid="stSidebar"] .stButton>button>div,[data-testid="stSidebar"] .stButton>button [data-has-shortcut]{width:100%;justify-content:flex-start!important}
[data-testid="stSidebar"] .stButton>button p{font-size:.83rem;font-weight:600;white-space:normal;text-align:left}
[data-testid="stSidebar"] .stButton>button:hover{background:#DCECDF;border-color:transparent;color:var(--ink)}
[data-testid="stSidebar"] .stButton>button[kind="primary"], [data-testid="stSidebar"] [data-testid="stBaseButton-primary"]{background:var(--primary)!important;color:white!important;border-color:var(--primary)!important}
[data-testid="stSidebar"] [data-testid="stBaseButton-primary"] p{color:#fff!important}
[data-testid="stSidebar"] [data-testid="stVerticalBlock"]{gap:8px}
.catalogue-card{padding:16px;background:var(--card);border:1px solid var(--border);border-radius:16px;margin:24px 0 8px}
.catalogue-head{display:flex;align-items:center;justify-content:space-between;gap:8px;color:var(--muted);font-size:.68rem;font-weight:700}
.data-badge{font-size:.6rem;line-height:1.5;font-weight:600;padding:3px 7px;border-radius:20px;color:var(--primary);background:var(--soft);white-space:nowrap}
.catalogue-stats{display:grid;grid-template-columns:1fr 1fr;margin-top:16px;gap:16px}
.catalogue-stats strong{font-size:1.5rem;line-height:1.2;display:block;letter-spacing:-.04em}
.catalogue-stats span{display:block;font-size:.7rem;color:var(--muted);margin-top:4px}
.sidebar-note{font-size:.69rem;line-height:1.65;color:var(--muted);margin:0 4px}
/* Inputs use one explicit surface and an accessible, visible keyboard focus ring. */
[data-testid="stSelectbox"] [data-baseweb="select"]>div,[data-testid="stMultiSelect"] [data-baseweb="select"]>div,[data-testid="stTextInput"] [data-baseweb="input"], [data-testid="stNumberInput"] [data-baseweb="input"], [data-testid="stSelectbox"] [data-testid="stSelectboxRoot"], [data-testid="stMultiSelect"] [data-testid="stMultiSelectRoot"]{background:var(--card)!important;border:1px solid var(--border)!important;border-radius:12px!important;min-height:44px;color:var(--ink)!important;box-shadow:0 2px 6px #193d2d03}
[data-testid="stSelectbox"] [role="combobox"], [data-testid="stMultiSelect"] [role="combobox"]{color:var(--ink)!important;font-weight:500;min-width:0}
[data-testid="stSelectbox"]:focus-within [data-baseweb="select"]>div,[data-testid="stMultiSelect"]:focus-within [data-baseweb="select"]>div,[data-testid="stTextInput"]:focus-within [data-baseweb="input"],[data-testid="stNumberInput"]:focus-within [data-baseweb="input"]{border-color:var(--primary)!important;outline:3px solid #0F7B5F33;outline-offset:2px}
button:focus-visible,input:focus-visible,[role="combobox"]:focus-visible,summary:focus-visible{outline:3px solid var(--primary)!important;outline-offset:3px!important}
input::placeholder{color:var(--muted)!important;opacity:1}
[data-testid="stTextInput"] input{background:var(--card)!important;border:1px solid var(--border)!important;border-radius:12px!important;min-height:44px;color:var(--ink)!important;padding:10px 12px!important}
[data-testid="stTextInput"] input:focus{border-color:var(--primary)!important;outline:3px solid #0F7B5F33!important;outline-offset:2px!important}
[data-testid="stNumberInput"] input{background:var(--card)!important;color:var(--ink)!important}
[data-testid="stMetricLabel"] p{font-weight:600!important;color:var(--muted)!important}
[data-testid="stSelectbox"] .react-aria-ComboBox>[role="group"],[data-testid="stMultiSelect"] [role="group"]{background:var(--card)!important;border:1px solid var(--border)!important;border-radius:12px!important;min-height:44px}
[data-testid="stSelectbox"] .react-aria-ComboBox>[role="group"]:focus-within,[data-testid="stMultiSelect"] [role="group"]:focus-within{border-color:var(--primary)!important;box-shadow:0 0 0 3px #0F7B5F33!important}
[data-testid="stTooltipIcon"] button{color:var(--muted)!important;opacity:1!important}
.stButton>button,.stDownloadButton>button{min-height:44px;border:1px solid var(--border);border-radius:12px;font-weight:600;color:var(--primary);background:white;transition:background .18s,transform .18s}
.stButton>button:hover,.stDownloadButton>button:hover{border-color:var(--primary);background:var(--soft);color:var(--primary)}
.selection{display:flex;align-items:center;gap:12px;padding:12px 16px;background:var(--card);border:1px solid var(--border);border-radius:16px;min-width:0}
.selection{margin-bottom:8px}.selection-body{flex:1;min-width:0}.selection-label{font-size:.62rem;text-transform:uppercase;letter-spacing:.1em;font-weight:700;color:var(--muted)}
.selection-name{font-size:.875rem;font-weight:600;line-height:1.45;overflow-wrap:anywhere}.selection-price{font-size:.9rem;font-weight:700;white-space:nowrap}
.stock-pill{color:var(--danger);background:var(--danger-bg);border:1px solid #EAC6BE;border-radius:24px;padding:5px 10px;font-size:.66rem;font-weight:700;white-space:nowrap}
.stock-pill.available{color:var(--amber);background:var(--amber-bg);border-color:#E6D5AC}
[data-testid="stExpander"]{background:var(--card)!important;border:1px solid var(--border)!important;border-radius:14px!important;box-shadow:0 2px 8px #193d2d02;min-width:0}
[data-testid="stExpander"] summary{font-size:.83rem;color:var(--ink);font-weight:600;padding:10px 16px!important;min-height:44px}
[data-testid="stExpander"] [data-testid="stExpanderDetails"]{padding:8px 16px 16px}
.product-grid{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:16px;animation:fade-in .3s ease-out}
.product-card{min-width:0;background:var(--card);border:1px solid var(--border);border-radius:16px;padding:16px;box-shadow:var(--shadow);transition:transform .18s ease,box-shadow .18s ease;position:relative}
.product-card:hover{transform:translateY(-4px);box-shadow:0 10px 24px #193D2D12}
.card-top{display:flex;align-items:center;gap:8px;min-width:0}.rank{font-size:.68rem;color:var(--muted);font-weight:700;font-variant-numeric:tabular-nums}
.tile-icon{width:36px;height:36px;display:inline-flex;align-items:center;justify-content:center;flex-shrink:0;background:var(--soft);border-radius:10px;font-size:1.4rem}
.badge{margin-left:auto;font-size:.6rem;font-weight:700;line-height:1.4;white-space:nowrap;padding:4px 8px;border-radius:24px;background:var(--soft);color:var(--primary)}
.badge.cheaper{background:var(--amber-bg);color:var(--amber)}
.product-card h3{font-size:.91rem!important;font-weight:600!important;line-height:1.45!important;letter-spacing:-.02em!important;margin:12px 0 4px!important;padding:0!important;min-height:42px;overflow-wrap:anywhere}
.category{font-size:.7rem;color:var(--muted);line-height:1.45;overflow-wrap:anywhere}
.price-row{display:flex;align-items:baseline;gap:8px;flex-wrap:wrap;margin-top:12px}.price{font-size:1.3rem;font-weight:700;letter-spacing:-.04em;color:var(--ink)}
.price-difference{color:var(--primary);font-size:.65rem;font-weight:600}.price-difference.higher{color:var(--amber)}
.score-label{display:flex;align-items:center;justify-content:space-between;gap:8px;font-size:.68rem;color:var(--muted);margin-top:12px;font-weight:500}.score-label b{color:var(--primary);font-weight:700}
.track{height:6px;background:#DCEBE2;border-radius:8px;margin:8px 0 0}.fill{background:var(--primary);height:6px;border-radius:8px}
.reason{color:var(--muted)!important;font-size:.7rem!important;font-weight:400!important;line-height:1.6!important;margin:10px 0 0!important;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
.empty-state{text-align:center;background:white;border:1px dashed #AFCBB9;border-radius:16px;padding:24px 16px 16px;margin-top:4px}
.empty-illustration{font-size:2.3rem;display:inline-flex;align-items:center;justify-content:center;width:72px;height:72px;background:var(--soft);border-radius:50%;margin-bottom:12px}
.empty-state h2{font-size:1.15rem!important;margin:0!important;padding:0!important}.empty-state p{color:var(--muted);font-size:.83rem;margin:8px 0 0}
.metric-card{background:var(--card);border:1px solid var(--border);border-radius:16px;padding:16px;box-shadow:var(--shadow);height:100%;min-width:0}
.metric-heading{font-size:.72rem;color:var(--muted);font-weight:600;line-height:1.5;display:flex;gap:8px;align-items:center}.metric-icon{font-size:1rem}.metric-value{font-size:1.8rem;line-height:1.2;font-weight:700;letter-spacing:-.05em;margin:12px 0 6px;color:var(--ink);overflow-wrap:anywhere}.metric-caption{font-size:.67rem;line-height:1.5;color:var(--muted)}
[data-testid="stMetric"]{background:var(--card);border:1px solid var(--border);border-radius:16px;padding:16px;box-shadow:var(--shadow)}
[data-testid="stMetricLabel"]{color:var(--muted)!important;font-weight:600}[data-testid="stMetricValue"]{color:var(--ink);font-size:1.8rem!important;font-weight:700}
.stAlert{border-radius:12px;color:var(--ink)}
[data-testid="stPlotlyChart"],[data-testid="stDataFrame"],[data-testid="stGraphVizChart"]{max-width:100%;min-width:0;overflow:hidden}
.footnote{border-top:1px solid var(--border);padding:16px 0 0;margin-top:24px;font-size:.66rem;line-height:1.6;color:var(--muted);text-align:center;overflow-wrap:anywhere}
.mobile-method{display:none}.method-step{padding:12px 16px;background:var(--card);border:1px solid var(--border);border-radius:12px;display:flex;align-items:center;gap:12px;font-size:.8rem;font-weight:600;color:var(--ink)}.method-step span{color:var(--primary);font-size:.65rem;font-weight:700}.method-arrow{color:var(--primary);text-align:center;line-height:1}
@keyframes fade-in{from{opacity:0;transform:translateY(6px)}to{opacity:1;transform:translateY(0)}}

@media(max-width:900px) and (min-width:641px){.product-card{padding:12px}.card-top{flex-wrap:wrap}.badge{font-size:.56rem;padding:4px 6px}}
@media(max-width:750px){[data-testid="stElementContainer"]:has([data-testid="stGraphVizChart"]),[data-testid="stGraphVizChart"]{display:none}.mobile-method{display:grid;gap:8px}.block-container{padding:3.5rem 1rem 1.5rem}.hero{padding:14px 16px}.hero h1{font-size:1.5rem!important;line-height:1.15!important}.hero p{font-size:.79rem}.eyebrow{font-size:.61rem}.selection{flex-wrap:wrap}.selection-body{flex-basis:calc(100% - 52px)}.selection-price{margin-left:48px}.stock-pill{margin-left:auto}.footnote{font-size:.6rem}.metric-value{font-size:1.65rem}.stApp{overflow-x:clip}[data-testid="stMain"]{min-width:0}h1{font-size:1.65rem!important}[data-testid="stHorizontalBlock"]{flex-wrap:wrap;gap:12px}[data-testid="stHorizontalBlock"]>[data-testid="stColumn"]{width:100%;flex:1 1 100%;min-width:0}.product-grid{grid-template-columns:1fr}.product-card h3{min-height:0!important}.metric-card{min-height:120px}}
@media(prefers-reduced-motion:reduce){*,*::before,*::after{animation:none!important;transition:none!important}.product-card:hover{transform:none}}

/* Section rhythm: prevent adjacent cards, controls and captions touching. */
[data-testid="stMain"] [data-testid="stVerticalBlock"]{gap:16px!important}
.hero{margin-bottom:12px}
.empty-state{margin-top:16px;margin-bottom:16px;padding:28px 20px 24px}
.empty-state p{margin-top:10px}
.metric-card{padding:20px;min-height:148px}
.metric-caption{margin-top:8px;line-height:1.65}
[data-testid="stMain"] [data-testid="stHorizontalBlock"]:has(.metric-card){margin-bottom:12px}
[data-testid="stMain"] [data-testid="stCaptionContainer"]{line-height:1.7;padding-top:4px;padding-bottom:4px}
[data-testid="stMain"] [data-testid="stHorizontalBlock"]:has([data-testid="stSelectbox"]){align-items:flex-end}
[data-testid="stToggle"]{padding:8px 0;min-height:44px;display:flex;align-items:center}
@media(max-width:750px){.empty-state{padding:24px 16px;margin-top:12px;margin-bottom:12px}.metric-card{min-height:132px}.hero{margin-bottom:8px}}
.score-ring{width:60px;height:60px;position:relative;margin:8px 0}.score-ring svg{width:60px;height:60px;transform:rotate(-90deg)}.score-ring circle{fill:none;stroke-width:4}.ring-track{stroke:#DCEBE2}.ring-value{stroke:var(--primary);stroke-dasharray:100;stroke-dashoffset:calc(100 - var(--score));stroke-linecap:round;animation:ring-reveal .8s ease-out}.score-ring b{position:absolute;inset:0;display:grid;place-items:center;font-size:.8rem;color:var(--ink)}@keyframes ring-reveal{from{stroke-dashoffset:100}}.local-explanation{font-size:.7rem;color:var(--muted);margin-top:8px}.local-explanation summary{cursor:pointer;font-weight:600;color:var(--primary)}.local-explanation p{white-space:normal;line-height:1.6}.local-explanation small{font-size:.65rem}.skeletons{display:grid;grid-template-columns:repeat(3,1fr);gap:16px}.skeleton{height:160px;border-radius:16px;background:linear-gradient(90deg,#EAF4EE,#fff,#EAF4EE);background-size:200% 100%;animation:shimmer 1s infinite}@keyframes shimmer{to{background-position:-200% 0}}@media(max-width:750px){.skeletons{grid-template-columns:1fr}.skeleton{height:80px}}
</style>""",unsafe_allow_html=True)

@st.cache_data(show_spinner=False)
def load_data(artifact_version):
    manifest=json.loads((ART/'manifest.json').read_text(encoding='utf-8'))
    tables={name:pd.read_csv(ART/f'{name}.csv') for name in ['products','rules','importance','model_metrics','ranking_metrics','mining_compare','switch_events','comparison']}
    curves=json.loads((ART/'curves.json').read_text(encoding='utf-8'))
    return manifest,tables,curves

@st.cache_resource(show_spinner=False)
def load_matrices(artifact_version):
    with np.load(ART/'matrices.npz',allow_pickle=False) as z:
        return {k:z[k] for k in z.files}

try:
    with st.spinner('Getting the shelves ready…'):
        ARTIFACT_VERSION=tuple((ART/name).stat().st_mtime_ns for name in ['manifest.json','matrices.npz','comparison.csv'])
        meta,tables,curves=load_data(ARTIFACT_VERSION); matrices=load_matrices(ARTIFACT_VERSION)
    products=tables['products']; names=products.name.to_dict()
except (OSError,ValueError,KeyError,json.JSONDecodeError) as exc:
    st.error('The catalogue is not ready. Run notebook sections 1–8 to create the artifacts, then refresh.')
    st.caption(f'Details: {type(exc).__name__}: {exc}')
    st.stop()

def esc(s): return html.escape(str(s))
def money(x): return f'${float(x):,.2f}'
def make_comparison_figure(frame,selected):
    # One interactive bar chart: the dropdown keeps unlike units off the same axis.
    metrics=['PR-AUC','ROC-AUC','F1','Precision@5','Recall@5','NDCG@5','MAP@5','Training time (s)','CV PR-AUC']
    labels=[('★ ' if name==selected else '')+('Soft Voting' if name=='Soft Voting (RF + GB + XGB)' else name) for name in frame['Model']]
    colors=['#0F7B5F' if name==selected else ('#4B7C69' if role=='Proposed' else '#5C7293')
            for name,role in zip(frame['Model'],frame['Role'])]
    fig=go.Figure()
    for i,metric in enumerate(metrics):
        values=frame[metric].astype(float).to_numpy()
        text=[f'{v:.2f}s' if metric=='Training time (s)' else f'{v:.4f}' for v in values]
        fig.add_trace(go.Bar(x=values,y=labels,orientation='h',name=metric,visible=i==0,
            marker_color=colors,text=text,textposition='outside',cliponaxis=False,customdata=frame['Model'].tolist(),
            hovertemplate='%{customdata}<br>'+metric+': %{x:.6f}<extra></extra>'))
    buttons=[]
    for i,metric in enumerate(metrics):
        maximum=float(frame[metric].max())
        buttons.append(dict(label=metric,method='update',args=[{'visible':[j==i for j in range(len(metrics))]},
            {'xaxis.title.text':metric,'xaxis.range':[0,max(1.15,maximum*1.22)],'title.text':'Model comparison · '+metric}]))
    fig.update_layout(title=dict(text='Model comparison · PR-AUC',x=0,y=.98,yanchor='top'),height=460,showlegend=False,meta={'comparison':True},
        font=dict(family='Inter',color='#4B5A52'),barcornerradius=5,
        xaxis=dict(title='PR-AUC',range=[0,1.15],showgrid=False,zeroline=False),
        yaxis=dict(autorange='reversed',automargin=True,showgrid=False),margin=dict(l=12,r=84,t=140,b=48),
        updatemenus=[dict(buttons=buttons,direction='down',x=1,y=1.15,xanchor='right',yanchor='top',
            bgcolor='white',bordercolor='#CFE3D8',font=dict(color='#193D2D'))],
        paper_bgcolor='rgba(0,0,0,0)',plot_bgcolor='rgba(0,0,0,0)')
    return fig

def chart(fig,key=None):
    # Presentation only: data, traces and evaluated values remain unchanged.
    heading=fig.layout.title.text or ''
    heading='<br>'.join(textwrap.wrap(heading,46))
    fig.update_layout(font=dict(family='Inter',size=12,color='#4B5A52'),
        title=dict(text=heading,font=dict(size=15,color='#193D2D'),x=0,xanchor='left'),
        margin=dict(l=12,r=84 if (fig.layout.meta or {}).get('comparison') else 12,t=140 if (fig.layout.meta or {}).get('comparison') else 68,b=48),paper_bgcolor='rgba(0,0,0,0)',plot_bgcolor='rgba(0,0,0,0)',
        legend_title_text='',legend=dict(orientation='h',yanchor='top',y=-.22,x=0,font=dict(size=11,color='#4B5A52')),
        barcornerradius=5,colorway=[ACCENT,'#99620B','#3E65A0','#AF4035'],
        coloraxis_colorscale=[[0,'#EAF4EE'],[1,ACCENT]],coloraxis_colorbar=dict(tickfont=dict(color='#4B5A52')))
    fig.update_xaxes(showgrid=False,zeroline=False,showline=False,automargin=True,tickfont=dict(color='#4B5A52'))
    fig.update_yaxes(showgrid=False,zeroline=False,showline=False,automargin=True,tickfont=dict(color='#4B5A52'))
    st.plotly_chart(fig,width='stretch',key=key,config={'displayModeBar':False,'responsive':True})

def metric_card(container,icon,label,value,caption,help_text=''):
    with container:
        st.markdown(f'<div class="metric-card" title="{esc(help_text)}"><div class="metric-heading"><span class="metric-icon" aria-hidden="true">{icon}</span>{esc(label)}</div><div class="metric-value">{esc(value)}</div><div class="metric-caption">{esc(caption)}</div></div>',unsafe_allow_html=True)

def set_page(name):
    st.session_state['freshmatch_page']=name

def choose_example(idx):
    st.session_state['chosen_product']=int(idx)

PRODUCT_ICONS={'Milk':'🥛','Cereal':'🥣','Bread':'🍞','Coffee':'☕','Pasta':'🍝','Sauce':'🥫','Protein':'🥡','Vegetables':'🥬','Snacks':'🥨','Dips':'🫙','Fruit':'🍎','Yogurt':'🥣'}

def title(kicker,heading,description):
    st.markdown(f'<div class="eyebrow">{esc(kicker)}</div><h1>{esc(heading)}</h1>',unsafe_allow_html=True)
    st.caption(description)

@st.cache_data(show_spinner=False,max_entries=512)
def recommend(query,max_price,same_brand,same_category,k,unavailable,artifact_version=ARTIFACT_VERSION):
    if query<0 or query>=len(products):
        return pd.DataFrame()
    q=products.iloc[query]
    mask=matrices['eligible'][query].copy()
    mask &= products.price.to_numpy()<=max_price+1e-8
    if same_brand: mask &= products.brand.to_numpy()==q.brand
    if same_category: mask &= products.category.to_numpy()==q.category
    for idx in unavailable:
        if 0<=idx<len(mask): mask[idx]=False
    ids=np.flatnonzero(mask)
    ids=ids[np.argsort(-matrices['scores'][query,ids],kind='stable')][:k]
    if not len(ids): return pd.DataFrame()
    out=products.iloc[ids].copy()
    out['score']=matrices['scores'][query,ids]*100
    out['buyer_context']=matrices['context'][query,ids]*100
    out['price_saving_pct']=(1-out.price/q.price)*100
    reasons=[]
    for idx,row in out.iterrows():
        similarity=float(matrices['context'][query,idx])*100
        location='Same aisle' if row.aisle==q.aisle else 'Same department'
        change=(1-row.price/q.price)*100
        price_words=f'{change:.0f}% cheaper' if change>=1 else ('similar price' if abs(change)<1 else f'{abs(change):.0f}% higher price')
        reasons.append(f'{location}, {similarity:.0f}% similar shopping patterns, {price_words}.')
    out['reason']=reasons
    friendly={'same_category':'same category','same_aisle':'same aisle','same_department':'same department','same_brand':'same brand','text_similarity':'name similarity','context_similarity':'shopping-context similarity','log_lift':'low co-purchase lift','price_ratio':'price similarity','popularity_ratio':'popularity balance','reorder_similarity':'reorder similarity'}
    explanations=[]
    for idx in out.index:
        effects=matrices['local_effects'][query,idx]
        top=np.argsort(-np.abs(effects))[:2]
        explanations.append('; '.join(f"{friendly[str(matrices['feature_names'][j])]} {effects[j]*100:+.1f} score points" for j in top))
    out['model_explanation']=explanations
    return out

def cards(result,original_price):
    if result.empty:
        st.info('No alternatives fit these filters. Raise the price limit or switch off a brand/category filter.')
        return
    rendered=[]
    for rank,(_,row) in enumerate(result.iterrows()):
        badge='Best match' if rank==0 else ('Cheaper' if row.price_saving_pct>1 else ('Popular' if row.popularity>=products.popularity.quantile(.7) else 'Alternative'))
        difference=float(original_price)-float(row.price)
        delta=f'↓ {money(difference)} cheaper' if difference>.005 else (f'↑ {money(-difference)} more' if difference<-.005 else 'Same price')
        direction='higher' if difference<-.005 else ''
        badge_style='cheaper' if badge=='Cheaper' else ''
        rendered.append(f"""<article class="product-card"><div class="card-top"><span class="rank">{rank+1:02d}</span><span class="tile-icon" aria-hidden="true">{PRODUCT_ICONS.get(row.category,'🛒')}</span><span class="badge {badge_style}">{badge}</span></div><h3>{esc(row['name'])}</h3><div class="category">{esc(row.brand)} · {esc(row.category)}</div><div class="price-row"><span class="price">{money(row.price)}</span><span class="price-difference {direction}">{delta}</span></div><div class="score-label"><span>Match score</span><b>{row.score:.0f} / 100</b></div><div class="score-ring" role="meter" aria-label="Match score" aria-valuemin="0" aria-valuemax="100" aria-valuenow="{row.score:.0f}"><svg viewBox="0 0 44 44" aria-hidden="true"><circle class="ring-track" cx="22" cy="22" r="18"/><circle class="ring-value" cx="22" cy="22" r="18" pathLength="100" style="--score:{row.score:.1f}"/></svg><b>{row.score:.0f}</b></div><p class="reason" title="{esc(row.reason)}">{esc(row.reason)}</p><details class="local-explanation"><summary>Why this score?</summary><p>{esc(row.model_explanation)}</p><small>Change vs training-median feature values; sensitivity, not causal or additive.</small></details></article>""")
    st.markdown('<div class="product-grid">'+''.join(rendered)+'</div>',unsafe_allow_html=True)
    st.download_button('↓ Download alternatives',result[['name','category','brand','price','score','reason','model_explanation']].to_csv(index=False).encode(),file_name='freshmatch_alternatives.csv',mime='text/csv')

PAGES=['Find Substitutes','Out-of-Stock Simulator','Product Explorer','Relationship Explorer','Model Performance','About']
NAV_ICONS=['🔎','🛒','🧺','🔗','📊','🌱']
if st.session_state.get('freshmatch_page') not in PAGES:
    st.session_state['freshmatch_page']=PAGES[0]
with st.sidebar:
    st.markdown('<div class="brand"><span class="brand-icon" aria-hidden="true">🌿</span><span class="brand-name">Fresh<span>Match</span></span></div><div class="subbrand">GOOD ALTERNATIVES. FULLER BASKETS.</div><div class="nav-label">Your workspace</div>',unsafe_allow_html=True)
    for i,(name,icon) in enumerate(zip(PAGES,NAV_ICONS)):
        st.button(name,icon=icon,key=f'nav_{i}',type='primary' if name==st.session_state['freshmatch_page'] else 'secondary',width='stretch',on_click=set_page,args=(name,))
    data_label='Synthetic data' if meta['source']!='Instacart' else 'Instacart'
    st.markdown(f'<section class="catalogue-card"><div class="catalogue-head">YOUR CATALOGUE <span class="data-badge">{data_label}</span></div><div class="catalogue-stats"><div><strong>{len(products):,}</strong><span>products</span></div><div><strong>{products.category.nunique()}</strong><span>categories</span></div></div></section><p class="sidebar-note">{esc(meta["price_note"])}.<br>{esc(meta["brand_note"])}.</p>',unsafe_allow_html=True)
page=st.session_state['freshmatch_page']

if page=='Find Substitutes':
    st.markdown('<div class="eyebrow">YOUR NEXT BEST CHOICE</div><section class="hero"><h1>Out of stock? Still in good company.</h1><p>Thoughtful alternatives to your everyday essentials, in one quick search.</p></section>',unsafe_allow_html=True)
    left,right=st.columns([3,1.25],vertical_alignment='bottom')
    with left:
        q=st.selectbox('Find your product',options=list(names),format_func=lambda i:names[i],index=None,placeholder='Search by product or brand…',key='chosen_product',help='Start typing a product or brand to search the catalogue.')
    with right:
        oos=st.toggle('Out of stock',value=True,help='Exclude the selected item and show available alternatives immediately.')
    if q is None:
        st.markdown('<section class="empty-state"><div class="empty-illustration" aria-hidden="true">🛒</div><h2>A good swap is one search away.</h2><p>Choose an essential above, or try one of these favourites.</p></section>',unsafe_allow_html=True)
        examples=products.drop_duplicates('category').head(3)
        columns=st.columns(3)
        for col,(idx,row) in zip(columns,examples.iterrows()):
            col.button(row['name'],icon=PRODUCT_ICONS.get(row.category,'🛒'),key=f'example_{idx}',width='stretch',on_click=choose_example,args=(idx,))
    else:
        selected=products.iloc[q]
        status='Out of stock' if oos else 'Exploring alternatives'
        status_class='' if oos else 'available'
        st.markdown(f'<div class="selection"><span class="tile-icon" aria-hidden="true">{PRODUCT_ICONS.get(selected.category,"🛒")}</span><div class="selection-body"><div class="selection-label">Your selection</div><div class="selection-name">{esc(selected["name"])}</div></div><span class="selection-price">{money(selected.price)}</span><span class="stock-pill {status_class}">{status}</span></div>',unsafe_allow_html=True)
        with st.expander('Refine your alternatives',expanded=False):
            c1,c2,c3,c4=st.columns([2,1,1.2,.9],vertical_alignment='center')
            max_catalogue=round(float(products.price.max()),2)
            limit=c1.slider('Maximum price ($)',min_value=.1,max_value=max(1.,max_catalogue),value=max(1.,max_catalogue),step=.1)
            same_brand=c2.checkbox('Same brand',value=False,help=meta['brand_note'])
            same_category=c3.checkbox('Same category',value=True,help='Keep alternatives within the selected product category.')
            k=c4.selectbox('Results',[3,5,8,10],index=1)
        heading,info=st.columns([3,1],vertical_alignment='center')
        heading.subheader('A good swap starts here')
        info.caption('Match score ⓘ',help='Calibrated suitability from heuristic pair labels. This is not a purchase probability or a guarantee of dietary equivalence.')
        loading=st.empty()
        loading.markdown('<div class="skeletons"><div class="skeleton"></div><div class="skeleton"></div><div class="skeleton"></div></div>',unsafe_allow_html=True)
        with st.spinner('Finding your best alternatives…'):
            result=recommend(int(q),float(limit),same_brand,same_category,k,(int(q),))
        loading.empty()
        cards(result,float(selected.price))
        with st.expander('How it works'):
            st.write('We look for products that serve a similar shopping purpose. We compare names, shared purchase patterns and prices, then avoid products people usually buy together. A higher match score means stronger evidence of suitability in our training labels.')
            st.caption('Similar shopping patterns compare co-purchase contexts; they do not mean that a stated percentage of buyers are identical. Check ingredients, allergies, pack size and dietary needs before swapping.')
            st.caption('An unavailable item is excluded. Turning the stock toggle off lets you browse the same alternatives for price or preference reasons.')

elif page=='Out-of-Stock Simulator':
    title('PLAN FOR THE UNEXPECTED','Keep the basket moving.','Test several unavailable products together and explore a clearly labelled recovery scenario.')
    unavailable=st.multiselect('Unavailable products',list(names),default=[0,24],format_func=lambda i:names[i],help='All selected products are removed from every recommendation pool.')
    c1,c2,c3=st.columns(3)
    demand=c1.number_input('Units of demand per unavailable item',min_value=1,max_value=10000,value=10,step=1)
    budget=c2.number_input('Maximum alternative price ($)',min_value=.1,max_value=1000.,value=float(np.ceil(products.price.max())),step=.5)
    only_cat=c3.checkbox('Keep the same category',value=True)
    st.caption('Scenario estimate ⓘ',help='Recovered units = units of demand × measured conditional top-5 future-switch hit rate, for covered products. It assumes that rate applies to this scenario; it is not a causal forecast.')
    if not unavailable:
        st.info('Choose one or more unavailable products to see how the basket could recover.')
    else:
        results={q:recommend(q,float(budget),False,only_cat,5,tuple(sorted(unavailable))) for q in unavailable}
        coverage=sum(not r.empty for r in results.values())/len(unavailable)
        observed=tables['ranking_metrics'].query("model == 'FreshMatch'").iloc[0]
        assumption=st.slider('Assumed customer acceptance (%)',0,100,int(round(observed.hit_rate*100)),help='Default: conditional offline hit rate. Adjust this scenario assumption; it is not a calibrated acceptance estimate.')/100
        units=demand*sum(not r.empty for r in results.values())*assumption
        revenue=demand*assumption*sum(float(r.iloc[0].price) for r in results.values() if not r.empty)
        cols=st.columns(3)
        metric_card(cols[0],'🛒','Estimated units recovered',f'{units:,.1f}','From your acceptance assumption',help_text='Scenario only: covered demand multiplied by the chosen acceptance assumption.')
        metric_card(cols[1],'💰','Estimated revenue saved',money(revenue),'Scenario estimate · not actual sales',help_text='Scenario units multiplied by each first-ranked alternative price. Prices are synthetic or estimated; no inventory capacity model is used.')
        metric_card(cols[2],'✓','Alternative coverage',f'{coverage:.0%}','Unavailable items with a suitable swap',help_text='Share of unavailable products with at least one eligible, available alternative.')
        st.caption(f'Default offline hit rate: {observed.hit_rate:.1%} (95% household-bootstrap interval {observed.hit_ci_low:.1%}–{observed.hit_ci_high:.1%}). Inventory quantities and shopper acceptance are scenario assumptions.')
        exports=[]
        for q,r in results.items():
            with st.expander(names[q],expanded=len(unavailable)<=2):
                if r.empty: st.info('No available alternatives under this price cap. Increase the cap or broaden the category filter.')
                else:
                    st.dataframe(r[['name','price','score','reason']].rename(columns={'name':'Alternative','price':'Price ($)','score':'Match / 100','reason':'Why it fits'}),hide_index=True,width='stretch')
                    exports.append(r.assign(unavailable_product=names[q]))
        if exports:
            st.download_button('↓ Download scenario',pd.concat(exports).to_csv(index=False).encode(),file_name='stockout_scenario.csv',mime='text/csv')

elif page=='Product Explorer':
    title('GET TO KNOW YOUR SHELVES','Everyday essentials, explored.','Browse the catalogue and discover what shoppers return to.')
    categories=['All categories']+sorted(products.category.unique())
    c1,c2=st.columns([1,2])
    category=c1.selectbox('Category',categories)
    search=c2.text_input('Search products',placeholder='Try coffee, milk, or a brand…',max_chars=120)
    subset=products if category=='All categories' else products[products.category==category]
    if search: subset=subset[subset.name.str.contains(search,case=False,regex=False)]
    if subset.empty: st.info('No products match that search. Try a shorter name or a different category.')
    else:
        cols=st.columns(3)
        cols[0].metric('Products',len(subset))
        cols[1].metric('Typical price',money(subset.price.median()),help=meta['price_note'])
        cols[2].metric('Repeat purchase rate',f'{subset.reorder_rate.mean():.0%}',help='Mean product reorder rate in historical purchases, not a forecast.')
        a,b=st.columns(2)
        with a:
            top=subset.nlargest(10,'popularity').sort_values('popularity')
            chart(px.bar(top,x='popularity',y='name',orientation='h',title='Most purchased essentials',labels={'popularity':'Historical item purchases','name':'Product'}))
        with b:
            chart(px.scatter(subset,x='price',y='popularity',color='category',size='reorder_rate',hover_name='name',title='Price and purchase frequency',labels={'price':'Price ($)','popularity':'Historical purchases','reorder_rate':'Repeat rate'}))
        st.dataframe(subset[['name','category','brand','price','popularity','reorder_rate']].rename(columns={'name':'Product','category':'Category','brand':'Brand','price':'Price ($)','popularity':'Purchases','reorder_rate':'Repeat rate'}),hide_index=True,width='stretch')

elif page=='Relationship Explorer':
    title('THE BIGGER PICTURE','Better together. Or instead.','See which products share a purpose and which belong in the same basket.')
    q=st.selectbox('Explore a product',list(names),format_func=lambda i:names[i])
    c1,c2=st.columns(2)
    c1.caption('Complement ⓘ',help='Products purchased together more often than expected; direct lift above 1.5.')
    c2.caption('Substitute ⓘ',help='An eligible alternative ranked by the calibrated model; this is a predicted relationship.')
    subs=recommend(int(q),float(products.price.max())+.1,False,False,5,(int(q),))
    comp_idx=np.flatnonzero((matrices['lift'][q]>1.5)&(np.arange(len(products))!=q))
    comp_idx=comp_idx[np.argsort(-matrices['lift'][q,comp_idx])][:5]
    nodes=list(dict.fromkeys([q]+subs.index.tolist()+comp_idx.tolist()))
    positions={q:(0.,0.)}
    for i,node in enumerate(nodes[1:]):
        angle=2*np.pi*i/max(1,len(nodes)-1)
        positions[node]=(np.cos(angle),np.sin(angle))
    fig=go.Figure()
    for label,ids,color in [('Substitute',subs.index.tolist(),ACCENT),('Complement',comp_idx.tolist(),'#99620B')]:
        xs=[];ys=[]
        for node in ids: xs.extend([0,positions[node][0],None]);ys.extend([0,positions[node][1],None])
        fig.add_trace(go.Scatter(x=xs,y=ys,mode='lines',line=dict(color=color,width=3),name=label,hoverinfo='skip'))
    fig.add_trace(go.Scatter(x=[positions[i][0] for i in nodes],y=[positions[i][1] for i in nodes],mode='markers+text',text=[names[i][:15]+'…' if len(names[i])>15 else names[i] for i in nodes],textfont=dict(size=10,color='#193D2D'),textposition='top center',hovertext=[names[i] for i in nodes],hoverinfo='text',marker=dict(size=[25 if i==q else 16 for i in nodes],color=['#203D31' if i==q else '#b9d9c9' for i in nodes],line=dict(color='#0F7B5F',width=2)),showlegend=False))
    fig.update_layout(title='Product relationship network · hover, pan and zoom',height=430,xaxis=dict(visible=False,range=[-1.5,1.5]),yaxis=dict(visible=False,range=[-1.4,1.4]),legend=dict(orientation='h'))
    chart(fig)
    st.subheader('Patterns in the basket')
    c1,c2,c3=st.columns([1,2,1])
    algo=c1.selectbox('Mining algorithm',['Apriori','FP-Growth'],help='Two exact frequent-itemset mining methods, run at the same support threshold.')
    term=c2.text_input('Search rules',placeholder='Filter either product name…',max_chars=120)
    min_lift=c3.number_input('Minimum lift',min_value=0.,max_value=100.,value=1.,step=.1,help='Observed co-purchase divided by expected co-purchase. Above one suggests complements.')
    rr=tables['rules']; rr=rr[(rr.algorithm==algo)&(rr.lift>=min_lift)].copy()
    rr['Product']=rr.a.map(names); rr['Bought with']=rr.b.map(names)
    if term: rr=rr[rr.Product.str.contains(term,case=False,regex=False)|rr['Bought with'].str.contains(term,case=False,regex=False)]
    if rr.empty: st.info('No rules match these filters. Lower the minimum lift or clear the search.')
    else:
        st.caption('Support ⓘ · Confidence ⓘ',help='Support: fraction of all historical baskets containing both products. Confidence: fraction of antecedent baskets that contain the consequent.')
        st.dataframe(rr[['Product','Bought with','support','confidence','lift']].sort_values('lift',ascending=False),hide_index=True,width='stretch')
        chart(px.scatter(rr,x='confidence',y='lift',size='support',hover_name='Product',hover_data=['Bought with'],title=f'{algo} · strength of co-purchase rules',labels={'confidence':'Confidence','lift':'Lift','support':'Support'}))
    mc=tables['mining_compare']
    a,b=st.columns(2)
    with a: chart(px.bar(mc,x='algorithm',y='seconds',title='Mining runtime (original computation)',labels={'algorithm':'Algorithm','seconds':'Seconds'}))
    with b: chart(px.bar(mc,x='algorithm',y='rules',title='Rules discovered',labels={'algorithm':'Algorithm','rules':'Directed rules'}))

elif page=='Model Performance':
    title('EVIDENCE, NOT GUESSWORK','A closer look at the match.','Weak-label classification and independent next-order ranking measure different things.')
    st.info('Scores below come from '+meta['source']+'. Weak labels are heuristic; the next-order benchmark is conditional on observed switches. These are offline results, not verified sales recovery.')
    comparison=tables['comparison']
    st.success('★ Deployed: '+meta['selected_model']+' · highest training-CV PR-AUC')
    st.caption('Proposed methods: Random Forest and Gradient Boosting. Comparators: Logistic Regression, XGBoost, MLPClassifier and soft voting (RF + GB + XGBoost).')
    st.caption('Selection criterion ⓘ',help='All six share the same product-disjoint train/test split and the same three CV folds. Deployment uses mean training-CV average precision, never held-out pair labels or future purchases. Exact ties keep the declared model order.')
    st.subheader('Six models, one evaluation protocol')
    def highlight_winner(row):
        return ['background-color:#EAF4EE;color:#084D39;font-weight:600' if row['Model']==meta['selected_model'] else '' for _ in row]
    st.dataframe(comparison.style.apply(highlight_winner,axis=1),hide_index=True,width='stretch',column_config={
        c:st.column_config.NumberColumn(c,format='%.2f' if c=='Training time (s)' else '%.6f')
        for c in ['ROC-AUC','PR-AUC','F1','Precision@5','Recall@5','NDCG@5','MAP@5','Training time (s)','CV PR-AUC']})
    chart(make_comparison_figure(comparison,meta['selected_model']),key='all_model_comparison')
    st.caption('Use the chart dropdown to compare metrics. ROC/PR/F1 use untouched weak-label pairs; Precision@5 and NDCG@5 use identical held-out next-order events. Training time includes search, refit and calibration; cache hits retain original times. The winning model need not win every held-out metric.')
    st.download_button('↓ Download six-model comparison',comparison.to_csv(index=False).encode(),file_name='model_comparison.csv',mime='text/csv')
    model_names=tables['model_metrics'].model.tolist()
    selected=st.selectbox('Model diagnostics',model_names,index=model_names.index(meta['selected_model']))
    m=tables['model_metrics'].set_index('model').loc[selected]
    cols=st.columns(4)
    definitions={'roc_auc':('ROC-AUC','Ability to rank positive weak-labelled pairs above negative pairs across thresholds.'),'pr_auc':('PR-AUC','Average precision: area summary of precision and recall under class imbalance.'),'f1':('F1','Harmonic mean of precision and recall at probability threshold 0.5.'),'brier':('Brier loss','Mean squared probability error on weak labels; lower is better.')}
    for col,(key,(label,tip)),icon in zip(cols,definitions.items(),['◷','↗','◎','◉']):
        metric_card(col,icon,label,f'{m[key]:.3f}','Lower is better' if key=='brier' else 'Untouched weak-label test',help_text=tip)
    c1,c2=st.columns(2)
    for container,kind in [(c1,'ROC'),(c2,'PR')]:
        with container:
            fig=go.Figure()
            for name,cur in curves['curves'].items():
                fig.add_trace(go.Scatter(x=cur['fpr' if kind=='ROC' else 'recall'],y=cur['tpr' if kind=='ROC' else 'precision'],mode='lines',name=name))
            fig.update_layout(title=f'{kind} · untouched weak-label test',xaxis_title='False positive rate' if kind=='ROC' else 'Recall',yaxis_title='True positive rate' if kind=='ROC' else 'Precision')
            chart(fig)
    c1,c2=st.columns(2)
    with c1:
        chart(px.bar(tables['importance'],x='importance',y='feature',orientation='h',title='Deployed model · permutation feature importance',labels={'importance':'Decrease in test PR-AUC','feature':'Feature'}))
        st.caption('Feature importance ⓘ',help='Permutation importance measures the decrease in test PR-AUC after shuffling each feature, averaged over three repeats. It is post-selection, descriptive and not causal. This panel shows the deployed model, '+meta['selected_model']+'.')
    with c2:
        chart(px.imshow(curves['confusion'][selected],text_auto=True,x=['Predicted no','Predicted yes'],y=['Actual no','Actual yes'],title=selected+' · confusion matrix',color_continuous_scale='Greens'))
        st.caption('Confusion matrix ⓘ',help='Counts of true/false positive/negative predictions at threshold 0.5 on held-out pair groups.')
    st.subheader('Will a shopper buy one of the alternatives next?')
    rm=tables['ranking_metrics']
    st.caption('Ranking metrics ⓘ',help='At K=5: Precision = hits/5; Recall = hits/all relevant next-order products; NDCG rewards higher positions; MAP averages precision at relevant ranks divided by all targets. Hit rate = events with any target in the top 5.')
    chart(px.bar(rm.melt(id_vars='model',value_vars=['precision_at_k','recall_at_k','ndcg_at_k','map_at_k']),x='variable',y='value',color='model',barmode='group',title='Next-order ranking vs identical-pool baselines',labels={'variable':'Metric @ 5','value':'Score'}))
    st.dataframe(rm,hide_index=True,width='stretch')
    row=rm[rm.model=='FreshMatch'].iloc[0]
    st.caption(f'{int(row.events):,} observed switching events · future target eligible in {row.target_coverage:.1%} of events · top-5 hit rate {row.hit_rate:.1%}, 95% household-bootstrap interval {row.hit_ci_low:.1%}–{row.hit_ci_high:.1%}.')
    with st.expander('Evaluation details and limitations'):
        for limitation in meta['limitations']: st.write('• '+limitation)
        st.write('Training and test pairs have disjoint unordered pair IDs. Products may occur in both. Feature construction uses only historical baskets; final orders are held out. Model selection uses training CV, not test scores. Future switching events need at least one observed alternative, so the measured hit rate should not be generalized to every stockout.')
        st.write('Prices are estimates; offline revenue is a proxy. All candidate products compete without inserting the future target. The same candidate policy is applied to all three ranking systems.')
    st.download_button('↓ Download ranking evaluation',rm.to_csv(index=False).encode(),file_name='ranking_metrics.csv',mime='text/csv')

elif page=='About':
    title('PURPOSE BUILT','A better alternative to an empty basket.','AI-Powered Product Substitution Discovery and Intelligent Alternative Recommendation Using Machine Learning')
    st.markdown('When a product is unavailable or too expensive, shoppers need something that fills the same need. FreshMatch combines product information with purchase patterns to offer useful alternatives and explain each choice.')
    st.subheader('From basket history to a better swap')
    st.graphviz_chart('digraph { rankdir=LR; node [shape=box,style="rounded,filled",fillcolor="#EDF4ED",fontname="Arial",color="#CFE0D2"]; A [label="History + products"]; B [label="Complement rules"]; C [label="Context + text + price"]; D [label="Weak pair labels"]; E [label="Tune + calibrate"]; F [label="Available alternatives"]; A -> B; A -> C; B -> C; C -> D; D -> E; E -> F; }')
    st.markdown('<div class="mobile-method" role="img" aria-label="Historical baskets and products feed complement rules and context, text and price features; weak pair labels train and calibrate models that rank available alternatives."><div class="method-step"><span>01</span>History + products</div><div class="method-arrow" aria-hidden="true">↓</div><div class="method-step"><span>02</span>Complement rules</div><div class="method-arrow" aria-hidden="true">↓</div><div class="method-step"><span>03</span>Context + text + price</div><div class="method-arrow" aria-hidden="true">↓</div><div class="method-step"><span>04</span>Weak pair labels</div><div class="method-arrow" aria-hidden="true">↓</div><div class="method-step"><span>05</span>Tune + calibrate</div><div class="method-arrow" aria-hidden="true">↓</div><div class="method-step"><span>06</span>Available alternatives</div></div>',unsafe_allow_html=True)
    st.caption('Association mining ⓘ · Calibration ⓘ',help='Association mining identifies frequent co-purchases. Calibration maps model outputs onto observed weak-label frequencies; it does not calibrate purchase acceptance.')
    a,b=st.columns(2)
    with a:
        st.subheader('What makes it different')
        st.write('Shared purpose and low co-purchase, rather than co-purchase alone. Price-aware filtering, plain-language reasons, multiple-stockout handling and transparent comparisons against simple baselines.')
        st.subheader('Built with')
        st.write('Python · pandas · SciPy · scikit-learn · MLxtend · Plotly · Streamlit · TF-IDF · Apriori · FP-Growth · Random Forest · Gradient Boosting · Logistic Regression · XGBoost · MLPClassifier · Soft Voting')
    with b:
        st.subheader('The team')
        st.write('**K. Darshan Sai**  \nRA2311027010002')
        st.write('**J. Ranga Mokshagna**  \nRA2311027010003')
        st.write('**P. Bharat Sai**  \nRA2311027010007')
        st.subheader('Dataset transparency')
        st.write(meta['source'])
        st.caption(f"{meta['history_baskets']:,} historical baskets · {meta['future_baskets']:,} future baskets · seed {meta['seed']}")
        st.caption(meta['price_note']+'. '+meta['brand_note']+'.')
    with st.expander('Responsible interpretation'):
        for limitation in meta['limitations']: st.write('• '+limitation)
        st.write('No nutritional, allergen or pack-size equivalence is guaranteed. Verify the product before buying. Ingredient data are outside this dataset.')

st.markdown('<div class="footnote">FreshMatch · K. Darshan Sai · J. Ranga Mokshagna · P. Bharat Sai</div>',unsafe_allow_html=True)


Overwriting app.py


In [11]:
config_dir=ROOT/'.streamlit'; config_dir.mkdir(exist_ok=True)
(config_dir/'config.toml').write_text("""[theme]
base="light"
primaryColor="#0F7B5F"
backgroundColor="#F6FAF7"
secondaryBackgroundColor="#F0F4EF"
textColor="#20352E"
font="sans serif"
[server]
headless=true
[browser]
gatherUsageStats=false
""",encoding='utf-8')
compile((ROOT/'app.py').read_text(encoding='utf-8'),'app.py','exec')
print('App source compiled successfully.')

App source compiled successfully.


### 10 · Launch Streamlit and print the public URL
On Colab/Linux or Windows this cell downloads Cloudflare's tunnel client over HTTPS and starts a temporary tunnel. The public URL is printed as soon as it appears in the log. Keep this runtime alive. The process binds to port 8501; rerunning reuses a healthy server and terminates only a tunnel previously created by this notebook. Set `FRESHMATCH_SKIP_LAUNCH=1` only for automated notebook validation.

The preview URL is public to anyone who has the link. This app exposes the retained product catalogue and aggregate metrics, not raw customer/order IDs. Temporary tunnels are best-effort and require outbound network access; a clear diagnostic is printed if the provider is unavailable.

In [12]:
import platform, re, urllib.request
def healthy():
    try:
        with urllib.request.urlopen('http://127.0.0.1:8501/_stcore/health',timeout=2) as response:
            return response.status==200
    except Exception:
        return False

if os.environ.get('FRESHMATCH_SKIP_LAUNCH')=='1':
    print('App launch skipped by automated validation configuration.')
else:
    if not healthy():
        server_log=open(ROOT/'streamlit.log','w',encoding='utf-8')
        streamlit_process=subprocess.Popen([sys.executable,'-m','streamlit','run',str(ROOT/'app.py'),
            '--server.port=8501','--server.address=0.0.0.0','--server.headless=true'],
            stdout=server_log,stderr=subprocess.STDOUT)
        for _ in range(40):
            if healthy(): break
            if streamlit_process.poll() is not None: break
            time.sleep(.5)
    if not healthy():
        raise RuntimeError('Streamlit did not start. Inspect streamlit.log and verify port 8501 is free.')
    print('Streamlit is healthy: http://localhost:8501')
    if platform.system() in ('Linux','Windows'):
        try:
            arch='arm64' if platform.machine() in ('aarch64','arm64') else 'amd64'
            suffix='.exe' if platform.system()=='Windows' else ''
            binary=ROOT/('cloudflared'+suffix)
            if not binary.exists():
                import requests
                target=f'cloudflared-windows-{arch}.exe' if suffix else f'cloudflared-linux-{arch}'
                response=requests.get(f'https://github.com/cloudflare/cloudflared/releases/latest/download/{target}',timeout=45)
                response.raise_for_status(); binary.write_bytes(response.content)
                if not suffix: binary.chmod(binary.stat().st_mode | 0o111)
            if 'tunnel_process' in globals() and tunnel_process.poll() is None:
                tunnel_process.terminate(); tunnel_process.wait(timeout=10)
            tunnel_log_path=ROOT/'tunnel.log'
            tunnel_log=open(tunnel_log_path,'w',encoding='utf-8')
            tunnel_process=subprocess.Popen([str(binary),'tunnel','--url','http://127.0.0.1:8501','--no-autoupdate'],stdout=tunnel_log,stderr=subprocess.STDOUT,
                creationflags=subprocess.CREATE_NO_WINDOW if platform.system()=='Windows' else 0)
            public_url=None
            for _ in range(75):
                contents=tunnel_log_path.read_text(encoding='utf-8',errors='replace')
                match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',contents)
                if match: public_url=match.group(0); break
                if tunnel_process.poll() is not None: break
                time.sleep(.6)
            if public_url:
                display(Markdown(f'## 🌿 Open FreshMatch\n**[Launch the public app]({public_url})**\n\nKeep this Colab runtime running.'))
                print('PUBLIC APP URL:',public_url)
            else:
                print('Tunnel did not return a URL. Last diagnostic:\n'+contents[-1600:])
                print('Rerun this cell after checking outbound network access. Local Streamlit is still running.')
        except Exception as exc:
            print(f'Temporary public tunnel unavailable: {type(exc).__name__}: {exc}')
            print('Local app is running; rerun this cell to retry the tunnel.')
    else:
        print('Local preview ready. Run this notebook in Colab/Linux or Windows for the automatic public tunnel.')
print(f'Total elapsed since imports: {perf_counter()-START:.1f} seconds')

App launch skipped by automated validation configuration.
Total elapsed since imports: 21.5 seconds


### 11 — Submission integrity checks
These checks run inside the notebook and fail if product separation, eligibility, score provenance or the saved comparison table drifts.

In [13]:
assert not endpoints(train_idx) & endpoints(test_idx)
for fold_train,fold_valid in cv:
    assert not endpoints(train_idx[fold_train]) & endpoints(train_idx[fold_valid])
assert np.all((cat[:,None]==cat[None,:])[eligible])
assert np.all(lift[eligible]<1.5) and not np.diag(eligible).any()
assert np.isfinite(local_effects).all()
assert comparison.loc[comparison.Deployed,'Model'].tolist()==[selected_name]
assert {'ROC-AUC','PR-AUC','F1','Precision@5','Recall@5','NDCG@5','MAP@5','Training time (s)'}<=set(comparison)
qa,qb=np.where(eligible)
with threadpool_limits(limits=1):
    expected_scores=model.predict_proba(pair_features(qa,qb))[:,1]
assert np.allclose(score[qa,qb],expected_scores)
pd.testing.assert_frame_equal(pd.read_csv(ART/'comparison.csv'),comparison,check_dtype=False)
assert len(models)==6 and all(len(np.unique(y[train_idx][v]))==2 for _,v in cv)
print('PASS: product-disjoint outer/CV splits, eligibility, local explanations, calibrated score provenance and shared six-model table.')


PASS: product-disjoint outer/CV splits, eligibility, local explanations, calibrated score provenance and shared six-model table.


### 12 · Submission checklist and reproducibility notes
Outputs: `artifacts/` (versioned catalogue, scores, models, rules and metrics), `RESULTS.md`, runtime-generated `app.py`, `.streamlit/config.toml`. Submit this notebook with `requirements.txt` and `README.md`; include `PROJECT_REPORT.md` and `PRESENTATION_OUTLINE.md` for your viva. Clear `.cache_freshmatch/` to time a cold run. Do not change the seed or thresholds after inspecting test results and then present the same test as untouched. A real deployment needs expert substitute labels, ingredient/pack-size constraints, stock quantities and a prospective randomized acceptance test.